# ROGII unified Kaggle pipeline

1. Tabular baseline with **well-group GroupKFold**
2. If **CV RMSE ≥ 12 ft**, run **episodic TCN** (checkpoints + manifest)
3. **Blend** tabular + episodic; optional tabular refine
4. **Auto-submit** to the competition via `kaggle competitions submit`


In [ ]:
from pathlib import Path
import json
ROOT = Path.cwd()
payload = json.loads("{\"run_kaggle_pipeline.py\": \"#!/usr/bin/env python3\\n\\\"\\\"\\\"Kaggle pipeline: baseline \\u2192 RMSE gate (12 ft) \\u2192 episodic TCN \\u2192 ensemble \\u2192 auto-submit.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport os\\nimport shutil\\nimport subprocess\\nimport sys\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\n\\n_BUNDLE = Path(__file__).resolve().parent\\nif str(_BUNDLE) not in sys.path:\\n    sys.path.insert(0, str(_BUNDLE))\\n\\nfrom run_baseline import resolve_data_dir, run as run_baseline, synthetic_oracle_rmse  # noqa: E402\\nfrom run_episodic_kaggle import run_episodic  # noqa: E402\\n\\nRMSE_FEET_THRESHOLD = 12.0\\nCOMPETITION_SLUG = \\\"rogii-wellbore-geology-prediction\\\"\\n\\n\\ndef cumulative_rmse_feet(fold_rmses: list[float]) -> float:\\n    \\\"\\\"\\\"Sum of per-fold OOF RMSE values (feet); used alongside mean cv_rmse for the gate.\\\"\\\"\\\"\\n    return float(sum(fold_rmses))\\n\\n\\ndef _validate_submission(submission: pd.DataFrame, sample_path: Path) -> dict:\\n    sample = pd.read_csv(sample_path)\\n    checks: list[dict] = []\\n    ok = True\\n    if len(submission) != len(sample):\\n        ok = False\\n        checks.append({\\\"name\\\": \\\"row_count\\\", \\\"status\\\": \\\"fail\\\"})\\n    else:\\n        checks.append({\\\"name\\\": \\\"row_count\\\", \\\"status\\\": \\\"pass\\\"})\\n    if list(submission.columns) != list(sample.columns):\\n        ok = False\\n        checks.append({\\\"name\\\": \\\"columns\\\", \\\"status\\\": \\\"fail\\\"})\\n    else:\\n        checks.append({\\\"name\\\": \\\"columns\\\", \\\"status\\\": \\\"pass\\\"})\\n    if submission.isna().any().any():\\n        ok = False\\n        checks.append({\\\"name\\\": \\\"no_nans\\\", \\\"status\\\": \\\"fail\\\"})\\n    else:\\n        checks.append({\\\"name\\\": \\\"no_nans\\\", \\\"status\\\": \\\"pass\\\"})\\n    return {\\\"ok\\\": ok, \\\"checks\\\": checks}\\n\\n\\ndef blend_submissions(\\n    baseline_sub: pd.DataFrame,\\n    episodic_sub: pd.DataFrame,\\n    *,\\n    id_col: str,\\n    target_col: str,\\n    baseline_rmse: float,\\n    episodic_rmse: float,\\n) -> tuple[pd.DataFrame, dict]:\\n    \\\"\\\"\\\"Inverse-RMSE weighted blend of tabular baseline and episodic TCN.\\\"\\\"\\\"\\n    b = baseline_sub.copy()\\n    e = episodic_sub.copy()\\n    b[id_col] = b[id_col].astype(str)\\n    e[id_col] = e[id_col].astype(str)\\n    merged = b.merge(e, on=id_col, suffixes=(\\\"_base\\\", \\\"_epi\\\"))\\n    w_base = 1.0 / max(baseline_rmse, 1e-6)\\n    w_epi = 1.0 / max(episodic_rmse, 1e-6)\\n    w_sum = w_base + w_epi\\n    wb, we = w_base / w_sum, w_epi / w_sum\\n    pred = wb * merged[f\\\"{target_col}_base\\\"] + we * merged[f\\\"{target_col}_epi\\\"]\\n    out = pd.DataFrame({id_col: merged[id_col], target_col: pred})\\n    meta = {\\n        \\\"blend_weights\\\": {\\\"tabular_baseline\\\": wb, \\\"episodic_tcn\\\": we},\\n        \\\"baseline_rmse_feet\\\": baseline_rmse,\\n        \\\"episodic_rmse_feet\\\": episodic_rmse,\\n    }\\n    return out, meta\\n\\n\\ndef submit_to_competition(\\n    submission_path: Path,\\n    *,\\n    message: str,\\n    competition: str = COMPETITION_SLUG,\\n) -> dict:\\n    \\\"\\\"\\\"Submit via Kaggle CLI when available (notebook or login node with credentials).\\\"\\\"\\\"\\n    if not shutil.which(\\\"kaggle\\\"):\\n        return {\\\"submitted\\\": False, \\\"reason\\\": \\\"kaggle CLI not on PATH\\\"}\\n    if os.environ.get(\\\"KAGGLE_KERNEL_RUN_TYPE\\\") and not os.environ.get(\\\"KAGGLE_API_TOKEN\\\"):\\n        # In-notebook: write for competition UI pickup; optional CLI if creds exist\\n        pass\\n    cmd = [\\n        \\\"kaggle\\\",\\n        \\\"competitions\\\",\\n        \\\"submit\\\",\\n        \\\"-c\\\",\\n        competition,\\n        \\\"-f\\\",\\n        str(submission_path),\\n        \\\"-m\\\",\\n        message,\\n        \\\"-q\\\",\\n    ]\\n    try:\\n        proc = subprocess.run(cmd, capture_output=True, text=True, check=False, timeout=120)\\n        if proc.returncode != 0:\\n            return {\\n                \\\"submitted\\\": False,\\n                \\\"reason\\\": proc.stderr.strip() or proc.stdout.strip() or f\\\"exit {proc.returncode}\\\",\\n                \\\"command\\\": \\\" \\\".join(cmd),\\n            }\\n        return {\\\"submitted\\\": True, \\\"message\\\": message, \\\"stdout\\\": proc.stdout.strip()}\\n    except Exception as exc:\\n        return {\\\"submitted\\\": False, \\\"reason\\\": str(exc), \\\"command\\\": \\\" \\\".join(cmd)}\\n\\n\\ndef run_pipeline(\\n    data_dir: Path,\\n    work_dir: Path,\\n    *,\\n    rmse_threshold: float = RMSE_FEET_THRESHOLD,\\n    n_splits: int = 5,\\n    submit: bool = True,\\n    episodic_episodes: int = 2,\\n    episodic_epochs: int = 30,\\n) -> dict:\\n    work_dir.mkdir(parents=True, exist_ok=True)\\n    sample_path = data_dir / \\\"sample_submission.csv\\\"\\n\\n    print(\\\"=== Phase 1: tabular baseline (GroupKFold / well groups) ===\\\")\\n    baseline_metrics = run_baseline(data_dir, work_dir, n_splits=n_splits)\\n    cv_rmse = float(baseline_metrics[\\\"cv_rmse\\\"])\\n    fold_rmses = [float(x) for x in baseline_metrics.get(\\\"fold_rmses\\\", [])]\\n    cum_rmse = cumulative_rmse_feet(fold_rmses)\\n\\n    gate_metric = cv_rmse\\n    trigger_episodic = gate_metric >= rmse_threshold\\n    print(\\n        f\\\"RMSE gate: cv_rmse={cv_rmse:.4f} ft  cumulative_sum={cum_rmse:.4f} ft  \\\"\\n        f\\\"threshold={rmse_threshold}  trigger={trigger_episodic}\\\"\\n    )\\n\\n    baseline_sub = pd.read_csv(work_dir / \\\"submission.csv\\\")\\n    id_col = str(baseline_metrics.get(\\\"id_column\\\", \\\"id\\\"))\\n    target_col = str(baseline_metrics.get(\\\"target_column\\\", \\\"tvt\\\"))\\n    if sample_path.is_file():\\n        cols = list(pd.read_csv(sample_path, nrows=0).columns)\\n        id_col, target_col = cols[0], cols[1]\\n\\n    final_sub = baseline_sub\\n    pipeline_meta: dict = {\\n        \\\"rmse_threshold_feet\\\": rmse_threshold,\\n        \\\"baseline_cv_rmse_feet\\\": cv_rmse,\\n        \\\"baseline_cumulative_rmse_feet\\\": cum_rmse,\\n        \\\"episodic_triggered\\\": trigger_episodic,\\n        \\\"group_key\\\": baseline_metrics.get(\\\"group_key\\\"),\\n        \\\"cv_scheme\\\": baseline_metrics.get(\\\"cv_scheme\\\"),\\n    }\\n\\n    if trigger_episodic:\\n        print(\\\"=== Phase 2: episodic TCN (threshold exceeded) ===\\\")\\n        epi = run_episodic(\\n            data_dir,\\n            work_dir,\\n            n_splits=n_splits,\\n            episodes_per_fold=episodic_episodes,\\n            max_epochs=episodic_epochs,\\n        )\\n        epi_sub = pd.read_csv(work_dir / \\\"submission_episodic.csv\\\")\\n        epi_cum = float(epi[\\\"cumulative_rmse_feet\\\"])\\n        epi_oof = float(epi[\\\"oof_rmse_feet\\\"])\\n        use_blend = epi_cum < cum_rmse\\n        pipeline_meta[\\\"episodic\\\"] = epi\\n        pipeline_meta[\\\"ensemble_manifest\\\"] = str(work_dir / \\\"checkpoints\\\" / \\\"episode_manifest.json\\\")\\n\\n        if use_blend:\\n            print(\\n                f\\\"=== Phase 3: ensemble blend (episodic cumulative {epi_cum:.4f} < baseline {cum_rmse:.4f}) ===\\\"\\n            )\\n            final_sub, blend_meta = blend_submissions(\\n                baseline_sub,\\n                epi_sub,\\n                id_col=id_col,\\n                target_col=target_col,\\n                baseline_rmse=cv_rmse,\\n                episodic_rmse=epi_oof,\\n            )\\n            pipeline_meta[\\\"blend\\\"] = blend_meta\\n        else:\\n            print(\\n                f\\\"=== Episodic did not beat baseline cumulative RMSE \\\"\\n                f\\\"({epi_cum:.4f} >= {cum_rmse:.4f}); keeping tabular submission ===\\\"\\n            )\\n            pipeline_meta[\\\"blend_skipped\\\"] = {\\n                \\\"reason\\\": \\\"episodic_cumulative_rmse_not_better_than_baseline\\\",\\n                \\\"baseline_cumulative_rmse_feet\\\": cum_rmse,\\n                \\\"episodic_cumulative_rmse_feet\\\": epi_cum,\\n                \\\"episodic_oof_rmse_feet\\\": epi_oof,\\n            }\\n\\n        print(\\\"=== Phase 4: re-run tabular with ensemble-informed fallback (optional refine) ===\\\")\\n        refined = run_baseline(data_dir, work_dir / \\\"refine_tabular\\\", n_splits=n_splits)\\n        refined_rmse = float(refined[\\\"cv_rmse\\\"])\\n        refined_cum = float(refined.get(\\\"cumulative_rmse_feet\\\", cumulative_rmse_feet(refined.get(\\\"fold_rmses\\\", []))))\\n        if refined_rmse < cv_rmse and use_blend:\\n            print(f\\\"Refined tabular improved {cv_rmse:.4f} -> {refined_rmse:.4f}; re-blending\\\")\\n            refined_sub = pd.read_csv(work_dir / \\\"refine_tabular\\\" / \\\"submission.csv\\\")\\n            final_sub, blend_meta = blend_submissions(\\n                refined_sub,\\n                epi_sub,\\n                id_col=id_col,\\n                target_col=target_col,\\n                baseline_rmse=refined_rmse,\\n                episodic_rmse=epi_oof,\\n            )\\n            pipeline_meta[\\\"refined_tabular_cv_rmse_feet\\\"] = refined_rmse\\n            pipeline_meta[\\\"refined_tabular_cumulative_rmse_feet\\\"] = refined_cum\\n            pipeline_meta[\\\"blend\\\"] = blend_meta\\n        elif refined_rmse < cv_rmse:\\n            print(f\\\"Refined tabular improved {cv_rmse:.4f} -> {refined_rmse:.4f}; using refined tabular only\\\")\\n            final_sub = pd.read_csv(work_dir / \\\"refine_tabular\\\" / \\\"submission.csv\\\")\\n            pipeline_meta[\\\"refined_tabular_cv_rmse_feet\\\"] = refined_rmse\\n            pipeline_meta[\\\"refined_tabular_cumulative_rmse_feet\\\"] = refined_cum\\n            cv_rmse = refined_rmse\\n            cum_rmse = refined_cum\\n    else:\\n        print(\\\"=== Episodic training skipped (cv_rmse below threshold) ===\\\")\\n\\n    final_path = work_dir / \\\"submission.csv\\\"\\n    final_sub.to_csv(final_path, index=False)\\n\\n    val = _validate_submission(final_sub, sample_path)\\n    pipeline_meta[\\\"submission_validation\\\"] = val\\n    holdout = synthetic_oracle_rmse(\\n        final_sub,\\n        data_dir / \\\"submission_samples\\\" / \\\"submission.csv\\\",\\n        id_col=id_col,\\n        target_col=target_col,\\n    )\\n    pipeline_meta[\\\"holdout_rmse_synthetic\\\"] = holdout\\n\\n    submit_result: dict = {\\\"submitted\\\": False, \\\"reason\\\": \\\"submit=False\\\"}\\n    if submit:\\n        print(\\\"=== Phase 5: Kaggle competition submit ===\\\")\\n        msg = (\\n            f\\\"rogii-trace-unified cv={cv_rmse:.3f} \\\"\\n            f\\\"epi={'yes' if trigger_episodic else 'no'}\\\"\\n        )\\n        submit_result = submit_to_competition(final_path, message=msg)\\n        pipeline_meta[\\\"kaggle_submit\\\"] = submit_result\\n        if submit_result.get(\\\"submitted\\\"):\\n            print(\\\"Kaggle submit OK:\\\", submit_result.get(\\\"stdout\\\", \\\"\\\"))\\n        else:\\n            print(\\\"Kaggle submit skipped/failed:\\\", submit_result.get(\\\"reason\\\"))\\n\\n    out_metrics = {\\n        **baseline_metrics,\\n        **pipeline_meta,\\n        \\\"final_cv_rmse_feet\\\": cv_rmse,\\n        \\\"final_cumulative_rmse_feet\\\": cum_rmse,\\n        \\\"final_submission\\\": str(final_path),\\n    }\\n    metrics_path = work_dir / \\\"pipeline_metrics.json\\\"\\n    metrics_path.write_text(json.dumps(out_metrics, indent=2) + \\\"\\\\n\\\", encoding=\\\"utf-8\\\")\\n    print(f\\\"Wrote {metrics_path}\\\")\\n    return out_metrics\\n\\n\\ndef main() -> int:\\n    import argparse\\n\\n    ap = argparse.ArgumentParser(description=\\\"ROGII Kaggle pipeline with RMSE gate\\\")\\n    ap.add_argument(\\\"--data-dir\\\", type=Path, default=None)\\n    ap.add_argument(\\\"--work-dir\\\", type=Path, default=Path(\\\"/kaggle/working\\\"))\\n    ap.add_argument(\\\"--rmse-threshold\\\", type=float, default=RMSE_FEET_THRESHOLD)\\n    ap.add_argument(\\\"--n-splits\\\", type=int, default=5)\\n    ap.add_argument(\\\"--no-submit\\\", action=\\\"store_true\\\")\\n    ap.add_argument(\\\"--episodic-episodes\\\", type=int, default=2)\\n    ap.add_argument(\\\"--episodic-epochs\\\", type=int, default=30)\\n    args = ap.parse_args()\\n    try:\\n        data_dir = resolve_data_dir(args.data_dir)\\n    except FileNotFoundError as exc:\\n        print(str(exc), file=sys.stderr)\\n        return 1\\n    run_pipeline(\\n        data_dir,\\n        args.work_dir.resolve(),\\n        rmse_threshold=args.rmse_threshold,\\n        n_splits=args.n_splits,\\n        submit=not args.no_submit,\\n        episodic_episodes=args.episodic_episodes,\\n        episodic_epochs=args.episodic_epochs,\\n    )\\n    return 0\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    raise SystemExit(main())\\n\", \"run_baseline.py\": \"#!/usr/bin/env python3\\n\\\"\\\"\\\"Unified ROGII baseline: multi-well load, CV RMSE, Kaggle submission, synthetic holdout RMSE.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport sys\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\n\\n_BUNDLE = Path(__file__).resolve().parent\\nif str(_BUNDLE) not in sys.path:\\n    sys.path.insert(0, str(_BUNDLE))\\n\\nfrom pipeline.competition_data import load_competition_frames  # noqa: E402\\nfrom train_predict import (  # noqa: E402\\n    align_train_target_to_schema,\\n    build_feature_matrix,\\n    categorize_columns,\\n    cross_val_and_predict,\\n    rmse,\\n)\\nfrom pipeline.preprocessor import replace_sentinels_with_nan  # noqa: E402\\nfrom pipeline.target_diagnostician import recommend_log1p  # noqa: E402\\nfrom pipeline import well_group_detector as wgd  # noqa: E402\\nfrom train_predict import _effective_cv, _HAS_LGBM, _configure_preprocessor_output  # noqa: E402\\n\\n_SENTINELS = (-999, -999.25, -9999, -99999)\\n\\n\\ndef _submission_id_parts(sample_sub: pd.DataFrame, id_col: str) -> pd.DataFrame:\\n    sub = sample_sub.copy()\\n    if \\\"row_idx\\\" not in sub.columns and id_col in sub.columns:\\n        parts = sub[id_col].astype(str).str.split(\\\"_\\\", n=1, expand=True)\\n        sub[\\\"well_id\\\"] = parts[0]\\n        sub[\\\"row_idx\\\"] = pd.to_numeric(parts[1], errors=\\\"coerce\\\").fillna(0).astype(int)\\n    return sub\\n\\n\\ndef align_test_preds_to_submission(\\n    test_pred: np.ndarray,\\n    test_df: pd.DataFrame,\\n    sample_sub: pd.DataFrame,\\n    id_col: str,\\n    *,\\n    fallback: float,\\n) -> np.ndarray:\\n    te = test_df.copy()\\n    te[\\\"_pred\\\"] = np.asarray(test_pred, dtype=np.float64)[: len(te)]\\n    te[\\\"_row\\\"] = te.groupby(\\\"well_id\\\").cumcount()\\n    lookup = te.set_index([\\\"well_id\\\", \\\"_row\\\"])[\\\"_pred\\\"]\\n    sub = _submission_id_parts(sample_sub, id_col)\\n    return np.array(\\n        [\\n            float(lookup.get((r[\\\"well_id\\\"], int(r[\\\"row_idx\\\"])), fallback))\\n            for _, r in sub.iterrows()\\n        ],\\n        dtype=np.float64,\\n    )\\n\\n\\ndef synthetic_oracle_rmse(\\n    submission: pd.DataFrame,\\n    oracle_path: Path,\\n    *,\\n    id_col: str,\\n    target_col: str,\\n) -> float | None:\\n    if not oracle_path.is_file():\\n        return None\\n    oracle = pd.read_csv(oracle_path)\\n    sub = submission.copy()\\n    sub[id_col] = sub[id_col].astype(str)\\n    oracle[id_col] = oracle[id_col].astype(str)\\n    merged = sub.merge(oracle, on=id_col, suffixes=(\\\"_pred\\\", \\\"_true\\\"))\\n    if merged.empty:\\n        return None\\n    pred_col = f\\\"{target_col}_pred\\\" if f\\\"{target_col}_pred\\\" in merged.columns else target_col\\n    true_col = f\\\"{target_col}_true\\\" if f\\\"{target_col}_true\\\" in merged.columns else target_col\\n    if pred_col == true_col:\\n        cols = [c for c in merged.columns if c.endswith(\\\"_pred\\\")]\\n        pred_col = cols[0] if cols else target_col\\n        true_col = pred_col.replace(\\\"_pred\\\", \\\"_true\\\")\\n    return rmse(merged[true_col].values, merged[pred_col].values)\\n\\n\\ndef resolve_data_dir(explicit: Path | None) -> Path:\\n    if explicit is not None:\\n        return explicit.resolve()\\n    kaggle = Path(\\\"/kaggle/input/rogii-wellbore-geology-prediction\\\")\\n    if kaggle.is_dir():\\n        return kaggle\\n  # Dev fallback: rogii repo synthetic flat layout\\n    dev = Path(\\\"/lustre/work/sweeden/rogii/data\\\")\\n    if dev.is_dir() and (dev / \\\"sample_submission.csv\\\").is_file():\\n        return dev\\n    local = _BUNDLE / \\\"data\\\"\\n    if local.is_dir():\\n        return local.resolve()\\n    raise FileNotFoundError(\\\"No competition data directory found (set --data-dir)\\\")\\n\\n\\ndef run(\\n    data_dir: Path,\\n    work_dir: Path,\\n    *,\\n    n_splits: int = 5,\\n) -> dict:\\n    work_dir.mkdir(parents=True, exist_ok=True)\\n    train_df, test_df, sample_sub, id_col, target_col = load_competition_frames(data_dir)\\n    sample_sub = _submission_id_parts(sample_sub, id_col)\\n\\n    train_target = \\\"TVT\\\" if \\\"TVT\\\" in train_df.columns else target_col\\n    train_df = align_train_target_to_schema(train_df, train_target)\\n    target = train_target\\n\\n    y_raw = train_df[target].astype(np.float64).values\\n    log_rec = recommend_log1p(y_raw, skew_threshold=1.5)\\n    use_log1p = bool(log_rec[\\\"use_log1p\\\"])\\n    is_positive = bool(log_rec[\\\"strict_positivity\\\"][\\\"strict_positive\\\"])\\n    y_fit = np.log1p(np.clip(y_raw, a_min=0.0, a_max=None)) if use_log1p else y_raw\\n    y_original = y_raw\\n\\n    base_exclude = {target, \\\"well_id\\\", \\\"is_train\\\", id_col}\\n    feature_cols = [\\n        c\\n        for c in train_df.columns\\n        if c not in base_exclude and c in test_df.columns and pd.api.types.is_numeric_dtype(train_df[c])\\n    ]\\n    if not feature_cols:\\n        raise ValueError(\\\"No overlapping numeric feature columns between train and test\\\")\\n\\n    X_train = replace_sentinels_with_nan(train_df[feature_cols].copy(), sentinel_values=_SENTINELS)\\n    X_test = replace_sentinels_with_nan(test_df[feature_cols].copy(), sentinel_values=_SENTINELS)\\n\\n    num, low_c, high_c = categorize_columns(\\n        pd.concat([X_train, X_test], axis=0, ignore_index=True),\\n        id_col=id_col,\\n        target_cols=[target],\\n    )\\n    preprocessor = build_feature_matrix(\\n        X_train, numeric_cols=num, low_card_cols=low_c, high_card_cols=high_c\\n    )\\n    if _HAS_LGBM:\\n        _configure_preprocessor_output(preprocessor)\\n\\n    scheme, groups, n_splits_eff = _effective_cv(train_df, test_df, id_col=id_col, n_splits_req=n_splits)\\n    group_key = wgd.recommend_group_key(train_df, test_df, id_column=id_col)\\n\\n    cv_rmse, fold_rmses, test_mat, oof_fit = cross_val_and_predict(\\n        X_train,\\n        y_fit,\\n        y_original,\\n        X_test,\\n        preprocessor,\\n        scheme=scheme,\\n        groups=groups,\\n        n_splits=n_splits_eff,\\n        use_log1p=use_log1p,\\n    )\\n\\n    test_mean_fit = test_mat.mean(axis=1)\\n    if use_log1p:\\n        test_pred = np.expm1(np.clip(test_mean_fit, a_min=None, a_max=20.0))\\n        if is_positive:\\n            test_pred = np.clip(test_pred, a_min=0.0, a_max=None)\\n    else:\\n        test_pred = test_mean_fit\\n        if is_positive:\\n            test_pred = np.clip(test_pred, a_min=0.0, a_max=None)\\n\\n    pred_fallback = float(np.nanmean(y_raw))\\n    aligned = align_test_preds_to_submission(\\n        test_pred, test_df, sample_sub, id_col, fallback=pred_fallback\\n    )\\n\\n    out_df = pd.DataFrame({id_col: sample_sub[id_col].astype(str), target_col: aligned})\\n    out_cols = [c for c in sample_sub.columns if c in out_df.columns]\\n    out_df = out_df[out_cols]\\n    sub_path = work_dir / \\\"submission.csv\\\"\\n    out_df.to_csv(sub_path, index=False)\\n\\n    oracle_path = data_dir / \\\"submission_samples\\\" / \\\"submission.csv\\\"\\n    holdout_rmse = synthetic_oracle_rmse(out_df, oracle_path, id_col=id_col, target_col=target_col)\\n\\n    cumulative_rmse_feet = float(sum(fold_rmses)) if fold_rmses else float(\\\"nan\\\")\\n\\n    metrics = {\\n        \\\"cv_rmse\\\": cv_rmse,\\n        \\\"cv_rmse_feet\\\": cv_rmse,\\n        \\\"cumulative_rmse_feet\\\": cumulative_rmse_feet,\\n        \\\"fold_rmses\\\": fold_rmses,\\n        \\\"n_folds\\\": n_splits_eff,\\n        \\\"cv_scheme\\\": scheme,\\n        \\\"group_key\\\": group_key,\\n        \\\"id_column\\\": id_col,\\n        \\\"target_column\\\": target_col,\\n        \\\"backend\\\": \\\"lightgbm\\\" if _HAS_LGBM else \\\"sklearn.hist_gradient_boosting\\\",\\n        \\\"use_log1p\\\": use_log1p,\\n        \\\"n_train_rows\\\": len(train_df),\\n        \\\"n_test_rows\\\": len(test_df),\\n        \\\"n_submission_rows\\\": len(out_df),\\n        \\\"n_wells_train\\\": int(train_df[\\\"well_id\\\"].nunique()),\\n        \\\"n_wells_test\\\": int(test_df[\\\"well_id\\\"].nunique()),\\n        \\\"data_dir\\\": str(data_dir),\\n        \\\"holdout_rmse_synthetic\\\": holdout_rmse,\\n    }\\n    metrics_path = work_dir / \\\"metrics.json\\\"\\n    metrics_path.write_text(json.dumps(metrics, indent=2) + \\\"\\\\n\\\", encoding=\\\"utf-8\\\")\\n\\n    print(f\\\"OOF CV RMSE (original scale): {cv_rmse:.6f}\\\")\\n    if holdout_rmse is not None:\\n        print(f\\\"Synthetic holdout RMSE (submission vs oracle): {holdout_rmse:.6f}\\\")\\n    print(f\\\"Wrote {sub_path} ({len(out_df)} rows)\\\")\\n    print(f\\\"Wrote {metrics_path}\\\")\\n    return metrics\\n\\n\\ndef main() -> int:\\n    ap = argparse.ArgumentParser(description=\\\"ROGII unified baseline runner\\\")\\n    ap.add_argument(\\\"--data-dir\\\", type=Path, default=None)\\n    ap.add_argument(\\\"--work-dir\\\", type=Path, default=Path(\\\"/kaggle/working\\\"))\\n    ap.add_argument(\\\"--n-splits\\\", type=int, default=5)\\n    args = ap.parse_args()\\n    try:\\n        data_dir = resolve_data_dir(args.data_dir)\\n    except FileNotFoundError as exc:\\n        print(str(exc), file=sys.stderr)\\n        return 1\\n    run(data_dir, args.work_dir.resolve(), n_splits=args.n_splits)\\n    return 0\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    raise SystemExit(main())\\n\", \"run_episodic_kaggle.py\": \"#!/usr/bin/env python3\\n\\\"\\\"\\\"Episodic TCN + checkpoint manifest for Kaggle (multi-well, GroupKFold by well_id).\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport os\\nimport time\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.model_selection import GroupKFold\\n\\nfrom pipeline.competition_data import load_competition_frames\\nfrom pipeline.episodic_benchmark import EpisodicBenchmark\\nfrom pipeline.target_diagnostician import recommend_log1p\\nfrom pipeline import well_group_detector as wgd\\nfrom pipeline.temporal_cnn import (\\n    TemporalCNN,\\n    make_sequences,\\n    predict_windows,\\n    reassemble,\\n    rmse,\\n    train_one_fold,\\n)\\nfrom run_baseline import _submission_id_parts, align_test_preds_to_submission\\n\\n\\ndef _fit_target_transform(\\n    y_raw: np.ndarray,\\n    *,\\n    use_log1p: bool,\\n) -> tuple[np.ndarray, dict]:\\n    \\\"\\\"\\\"Map TVT (feet) to a zero-mean unit-variance training scale for the TCN head.\\\"\\\"\\\"\\n    y = np.asarray(y_raw, dtype=np.float64)\\n    if use_log1p:\\n        y = np.log1p(np.clip(y, 0.0, None))\\n    mu = float(np.mean(y))\\n    sd = float(np.std(y))\\n    if sd <= 0.0:\\n        sd = 1.0\\n    return (y - mu) / sd, {\\\"use_log1p\\\": use_log1p, \\\"target_mean\\\": mu, \\\"target_std\\\": sd}\\n\\n\\ndef _invert_target_transform(pred_scaled: np.ndarray, meta: dict) -> np.ndarray:\\n    \\\"\\\"\\\"Undo z-score (and optional log1p) back to feet.\\\"\\\"\\\"\\n    fit = np.asarray(pred_scaled, dtype=np.float64) * meta[\\\"target_std\\\"] + meta[\\\"target_mean\\\"]\\n    if meta[\\\"use_log1p\\\"]:\\n        fit = np.expm1(np.clip(fit, None, 20.0))\\n        return np.clip(fit, 0.0, None)\\n    return fit\\n\\n\\ndef _zscore_per_well(df: pd.DataFrame, cols: list[str], well_col: str = \\\"well_id\\\") -> pd.DataFrame:\\n    out = df.copy()\\n    for c in cols:\\n        g = out.groupby(well_col)[c]\\n        mu = g.transform(\\\"mean\\\")\\n        sd = g.transform(\\\"std\\\").replace(0, 1)\\n        out[c] = (out[c] - mu) / sd\\n    return out\\n\\n\\ndef _build_splits(\\n    train_n: pd.DataFrame,\\n    target_col: str,\\n    *,\\n    n_splits: int,\\n) -> tuple[list[tuple[np.ndarray, np.ndarray]], str, str | None]:\\n    \\\"\\\"\\\"GroupKFold on well_id when possible; else depth-block CV within wells.\\\"\\\"\\\"\\n    group_key = wgd.recommend_group_key(train_n, id_column=\\\"id\\\")\\n    n_wells = int(train_n[\\\"well_id\\\"].nunique())\\n    if n_wells >= 2 and group_key:\\n        groups = (\\n            wgd.provide_groups(train_n, group_key, id_column=\\\"id\\\")\\n            if not group_key.startswith(\\\"__derived\\\")\\n            else train_n[\\\"well_id\\\"].astype(str).values\\n        )\\n        n_eff = min(n_splits, wgd.count_unique_groups(groups))\\n        splits = list(\\n            GroupKFold(n_splits=n_eff).split(\\n                train_n, train_n[target_col], groups=groups\\n            )\\n        )\\n        scheme = f\\\"groupkfold(n_splits={n_eff})\\\"\\n        return splits, scheme, group_key if not group_key.startswith(\\\"__\\\") else \\\"well_id\\\"\\n\\n    qbins = pd.qcut(train_n[\\\"MD\\\"], q=n_splits, labels=False, duplicates=\\\"drop\\\")\\n    splits = []\\n    for fold in sorted(qbins.unique()):\\n        va = np.where(qbins == fold)[0]\\n        tr = np.where(qbins != fold)[0]\\n        splits.append((tr, va))\\n    return splits, f\\\"depth_block_cv(n_splits={n_splits})\\\", \\\"MD\\\"\\n\\n\\ndef run_episodic(\\n    data_dir: Path,\\n    work_dir: Path,\\n    *,\\n    n_splits: int = 5,\\n    episodes_per_fold: int = 2,\\n    max_epochs: int = 30,\\n    patience: int = 6,\\n    window_len: int = 128,\\n    stride: int = 64,\\n    hidden: int = 64,\\n    n_blocks: int = 4,\\n    batch_size: int = 64,\\n    seed: int = 42,\\n) -> dict:\\n    \\\"\\\"\\\"Train episodic TCN; write checkpoints, benchmark JSON, and submission CSV.\\\"\\\"\\\"\\n    import torch\\n\\n    work_dir.mkdir(parents=True, exist_ok=True)\\n    ckpt_dir = work_dir / \\\"checkpoints\\\"\\n    ckpt_dir.mkdir(parents=True, exist_ok=True)\\n\\n    train_df, test_df, sample_sub, id_col, target_col = load_competition_frames(data_dir)\\n    sample_sub = _submission_id_parts(sample_sub, id_col)\\n\\n    train_target = \\\"TVT\\\" if \\\"TVT\\\" in train_df.columns else target_col\\n    if train_target not in train_df.columns:\\n        by_lower = {str(c).lower(): c for c in train_df.columns}\\n        train_target = by_lower.get(train_target.lower(), train_target)\\n\\n    y_raw = train_df[train_target].astype(np.float64).values\\n    log_rec = recommend_log1p(y_raw, skew_threshold=1.5)\\n    use_log1p = bool(log_rec[\\\"use_log1p\\\"])\\n    y_fit, target_meta = _fit_target_transform(y_raw, use_log1p=use_log1p)\\n    train_df = train_df.copy()\\n    train_df[train_target] = y_fit\\n    print(\\n        \\\"episodic target: \\\"\\n        f\\\"{'log1p + ' if use_log1p else ''}global z-score \\\"\\n        f\\\"(mean={target_meta['target_mean']:.4f}, std={target_meta['target_std']:.4f})\\\"\\n    )\\n\\n    exclude = {train_target, \\\"well_id\\\", \\\"is_train\\\", id_col}\\n    feature_cols = [\\n        c\\n        for c in train_df.columns\\n        if c not in exclude and c in test_df.columns and pd.api.types.is_numeric_dtype(train_df[c])\\n    ]\\n    if \\\"MD\\\" not in feature_cols:\\n        raise ValueError(\\\"Episodic TCN requires MD in train and test\\\")\\n\\n    train_n = _zscore_per_well(train_df, feature_cols).sort_values([\\\"well_id\\\", \\\"MD\\\"]).reset_index(drop=True)\\n    test_n = _zscore_per_well(test_df, feature_cols).sort_values([\\\"well_id\\\", \\\"MD\\\"]).reset_index(drop=True)\\n\\n    splits, cv_scheme, group_key = _build_splits(train_n, train_target, n_splits=n_splits)\\n    overlap = wgd.check_train_test_overlap(\\n        train_n[\\\"well_id\\\"].astype(str).values,\\n        test_n[\\\"well_id\\\"].astype(str).values,\\n    )\\n\\n    X_test_seq, _, test_seq_map = make_sequences(\\n        test_n,\\n        well_col=\\\"well_id\\\",\\n        depth_col=\\\"MD\\\",\\n        feature_cols=feature_cols,\\n        target_col=None,\\n        window_len=window_len,\\n        stride=stride,\\n    )\\n\\n    device = torch.device(\\\"cuda\\\" if torch.cuda.is_available() else \\\"cpu\\\")\\n    print(f\\\"episodic device: {device}  cv: {cv_scheme}  group_key: {group_key}\\\")\\n\\n    benchmark = EpisodicBenchmark(\\n        variant=\\\"kaggle_unified\\\",\\n        approach=\\\"episodic TCN (Kaggle trigger)\\\",\\n        eval_mask=\\\"full_well\\\",\\n        use_log1p=use_log1p,\\n        feature_cols=feature_cols,\\n        episodes_per_fold=episodes_per_fold,\\n        max_epochs=max_epochs,\\n        patience=patience,\\n    )\\n    benchmark.cv_scheme = cv_scheme\\n\\n    oof = np.full(len(train_n), np.nan, dtype=np.float64)\\n    fold_rmses: list[float] = []\\n    test_preds_per_fold = np.zeros((len(test_n), len(splits)), dtype=np.float64)\\n    best_checkpoints: list[dict] = []\\n    t0 = time.time()\\n\\n    for fold, (tr_idx, va_idx) in enumerate(splits):\\n        best_ep_rmse = float(\\\"inf\\\")\\n        best_ep_path: Path | None = None\\n        for ep in range(episodes_per_fold):\\n            ep_seed = seed + fold * 1000 + ep * 17\\n            torch.manual_seed(ep_seed)\\n            np.random.seed(ep_seed)\\n\\n            tr_df = train_n.iloc[tr_idx].reset_index(drop=True)\\n            va_df = train_n.iloc[va_idx].reset_index(drop=True)\\n            X_tr, y_tr, _ = make_sequences(\\n                tr_df,\\n                well_col=\\\"well_id\\\",\\n                depth_col=\\\"MD\\\",\\n                feature_cols=feature_cols,\\n                target_col=train_target,\\n                window_len=window_len,\\n                stride=stride,\\n            )\\n            X_va, y_va, va_map = make_sequences(\\n                va_df,\\n                well_col=\\\"well_id\\\",\\n                depth_col=\\\"MD\\\",\\n                feature_cols=feature_cols,\\n                target_col=train_target,\\n                window_len=window_len,\\n                stride=stride,\\n            )\\n            model, best_win_rmse, hist = train_one_fold(\\n                X_tr,\\n                y_tr,\\n                X_va,\\n                y_va,\\n                n_features=len(feature_cols),\\n                hidden=hidden,\\n                n_blocks=n_blocks,\\n                max_epochs=max_epochs,\\n                patience=patience,\\n                batch_size=batch_size,\\n                verbose=False,\\n            )\\n            model = model.to(device)\\n\\n            ckpt_path = ckpt_dir / f\\\"fold_{fold}_ep_{ep}.pt\\\"\\n            torch.save(\\n                {\\n                    \\\"state_dict\\\": model.state_dict(),\\n                    \\\"fold\\\": fold,\\n                    \\\"episode\\\": ep,\\n                    \\\"val_rmse\\\": best_win_rmse,\\n                    \\\"feature_cols\\\": feature_cols,\\n                    \\\"window_len\\\": window_len,\\n                    \\\"hidden\\\": hidden,\\n                    \\\"n_blocks\\\": n_blocks,\\n                    \\\"target_meta\\\": target_meta,\\n                },\\n                ckpt_path,\\n            )\\n\\n            va_win = predict_windows(model, X_va, batch_size=batch_size)\\n            va_row = reassemble(va_win, va_map)\\n            va_fit = va_row[: len(va_df)]\\n            y_va_feet = y_raw[va_idx]\\n            ep_val_feet = rmse(y_va_feet, _invert_target_transform(va_fit, target_meta))\\n\\n            benchmark.record_episode(\\n                fold=fold,\\n                episode=ep,\\n                seed=ep_seed,\\n                val_rmse=ep_val_feet,\\n                best_window_rmse=best_win_rmse,\\n                checkpoint=str(ckpt_path.name),\\n                n_epochs=len(hist),\\n            )\\n            print(f\\\"  fold {fold} ep {ep}: val_rmse={ep_val_feet:.4f} ft\\\")\\n\\n            if ep_val_feet < best_ep_rmse:\\n                best_ep_rmse = ep_val_feet\\n                best_ep_path = ckpt_path\\n                oof[va_idx] = va_fit\\n\\n        if best_ep_path is None:\\n            raise RuntimeError(f\\\"No episodic checkpoint for fold {fold}\\\")\\n\\n        fold_rmses.append(best_ep_rmse)\\n        benchmark.set_fold_best(fold, best_ep_path.name, best_ep_rmse)\\n        best_checkpoints.append(\\n            {\\\"fold\\\": fold, \\\"checkpoint\\\": best_ep_path.name, \\\"val_rmse\\\": best_ep_rmse}\\n        )\\n\\n        ckpt = torch.load(best_ep_path, map_location=device, weights_only=False)\\n        model = TemporalCNN(\\n            len(feature_cols),\\n            hidden=hidden,\\n            n_blocks=n_blocks,\\n        ).to(device)\\n        model.load_state_dict(ckpt[\\\"state_dict\\\"])\\n        te_win = predict_windows(model, X_test_seq, batch_size=batch_size)\\n        test_preds_per_fold[:, fold] = reassemble(te_win, test_seq_map)\\n\\n    elapsed = time.time() - t0\\n    mask = ~np.isnan(oof)\\n    oof_feet = _invert_target_transform(oof[mask], target_meta)\\n    oof_rmse = rmse(y_raw[mask], oof_feet)\\n    cumulative_rmse_feet = float(sum(fold_rmses))\\n\\n    test_pred = _invert_target_transform(test_preds_per_fold.mean(axis=1), target_meta)\\n    fallback = float(np.mean(y_raw))\\n    aligned = align_test_preds_to_submission(\\n        test_pred, test_n, sample_sub, id_col, fallback=fallback\\n    )\\n    sub = pd.DataFrame({id_col: sample_sub[id_col].astype(str), target_col: aligned})\\n    out_cols = [c for c in sample_sub.columns if c in sub.columns]\\n    sub = sub[out_cols]\\n    sub_path = work_dir / \\\"submission_episodic.csv\\\"\\n    sub.to_csv(sub_path, index=False)\\n\\n    benchmark.finalize(\\n        oof_rmse=oof_rmse,\\n        fold_rmses=fold_rmses,\\n        elapsed_seconds=elapsed,\\n        oof_rmse_raw_scale=oof_rmse,\\n    )\\n    benchmark.write_json(work_dir / \\\"episodic_benchmark.json\\\")\\n\\n    manifest = {\\n        \\\"cv_scheme\\\": cv_scheme,\\n        \\\"group_key\\\": group_key,\\n        \\\"well_group_overlap\\\": overlap,\\n        \\\"target_transform\\\": target_meta,\\n        \\\"oof_rmse_feet\\\": oof_rmse,\\n        \\\"cumulative_rmse_feet\\\": cumulative_rmse_feet,\\n        \\\"fold_rmses\\\": fold_rmses,\\n        \\\"best_checkpoints\\\": best_checkpoints,\\n        \\\"checkpoints_dir\\\": str(ckpt_dir),\\n        \\\"elapsed_seconds\\\": elapsed,\\n        \\\"device\\\": str(device),\\n    }\\n    (ckpt_dir / \\\"episode_manifest.json\\\").write_text(\\n        json.dumps(manifest, indent=2) + \\\"\\\\n\\\", encoding=\\\"utf-8\\\"\\n    )\\n    (work_dir / \\\"ensemble_manifest.json\\\").write_text(\\n        json.dumps(\\n            {\\n                \\\"models\\\": [\\\"tabular_baseline\\\", \\\"episodic_tcn\\\"],\\n                \\\"checkpoints\\\": best_checkpoints,\\n                \\\"episodic_oof_rmse_feet\\\": oof_rmse,\\n            },\\n            indent=2,\\n        )\\n        + \\\"\\\\n\\\",\\n        encoding=\\\"utf-8\\\",\\n    )\\n\\n    print(f\\\"Episodic OOF RMSE: {oof_rmse:.4f} ft  cumulative_fold_sum: {cumulative_rmse_feet:.4f}\\\")\\n    print(f\\\"Wrote {sub_path} and checkpoints under {ckpt_dir}\\\")\\n\\n    return {\\n        \\\"oof_rmse_feet\\\": oof_rmse,\\n        \\\"cumulative_rmse_feet\\\": cumulative_rmse_feet,\\n        \\\"fold_rmses\\\": fold_rmses,\\n        \\\"cv_scheme\\\": cv_scheme,\\n        \\\"group_key\\\": group_key,\\n        \\\"target_transform\\\": target_meta,\\n        \\\"submission_path\\\": str(sub_path),\\n        \\\"checkpoints\\\": best_checkpoints,\\n        \\\"elapsed_seconds\\\": elapsed,\\n    }\\n\", \"train_predict.py\": \"#!/usr/bin/env python3\\n\\\"\\\"\\\"RMSE-oriented baseline for Kaggle *ROGII - Wellbore Geology Prediction*.\\n\\nLeaderboard metric::\\n\\n    RMSE = sqrt( (1/n) * sum_i (y_hat_i - y_i)^2 )\\n\\nTraining may use ``log1p(y)`` when the target is strictly positive and skewed; local\\n**OOF RMSE** is reported on the **original target scale** (after ``expm1`` when\\napplicable) to match Kaggle. Prefer **GroupKFold** on a well-like column or on a\\nprefix derived from ``id`` when no dedicated well column exists.\\n\\nUsage::\\n\\n    python train_predict.py --data-dir data --out submission.csv\\n\\nDependencies: pandas, numpy, scikit-learn; optional lightgbm (recommended).\\nRun from the competition directory (the folder that contains ``pipeline/``).\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport sys\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.base import clone\\nfrom sklearn.compose import ColumnTransformer\\nfrom sklearn.impute import SimpleImputer\\nfrom sklearn.pipeline import Pipeline\\nfrom sklearn.preprocessing import OneHotEncoder, OrdinalEncoder\\n\\n_ROOT = Path(__file__).resolve().parent\\nif str(_ROOT) not in sys.path:\\n    sys.path.insert(0, str(_ROOT))\\n\\nfrom pipeline.cv_orchestrator import choose_cv_scheme, emit_fold_indices\\nfrom pipeline.nb_support import ensure_id_column\\nfrom pipeline.preprocessor import replace_sentinels_with_nan\\nfrom pipeline.target_diagnostician import recommend_log1p\\nfrom pipeline import well_group_detector as wgd\\n\\ntry:\\n    import lightgbm as lgb\\n\\n    _HAS_LGBM = True\\nexcept ImportError:\\n    _HAS_LGBM = False\\n\\nfrom sklearn.ensemble import HistGradientBoostingRegressor\\n\\n_SENTINELS = (-999, -999.25, -9999, -99999)\\n\\n\\ndef rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:\\n    \\\"\\\"\\\"Root mean squared error (matches Kaggle definition for vector errors).\\\"\\\"\\\"\\n    y_true = np.asarray(y_true, dtype=np.float64).ravel()\\n    y_pred = np.asarray(y_pred, dtype=np.float64).ravel()\\n    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))\\n\\n\\ndef _find_default_csv(data_dir: Path, stem: str) -> Path:\\n    for name in (f\\\"{stem}.csv\\\", f\\\"{stem.capitalize()}.csv\\\"):\\n        p = data_dir / name\\n        if p.is_file():\\n            return p\\n    for p in sorted(data_dir.glob(\\\"*.csv\\\")):\\n        if p.name.startswith(\\\".\\\") or p.name.startswith(\\\"._\\\"):\\n            continue\\n        if stem.lower() in p.stem.lower():\\n            return p\\n    raise FileNotFoundError(f\\\"No {stem} CSV under {data_dir}\\\")\\n\\n\\ndef infer_columns(sample_submission: pd.DataFrame) -> tuple[str, list[str]]:\\n    cols = list(sample_submission.columns)\\n    if len(cols) < 2:\\n        raise ValueError(\\\"sample_submission needs at least id + one target column\\\")\\n    id_col = cols[0]\\n    target_cols = cols[1:]\\n    return id_col, target_cols\\n\\n\\ndef align_train_target_to_schema(train_df: pd.DataFrame, target_name: str) -> pd.DataFrame:\\n    \\\"\\\"\\\"Rename train target column to match ``sample_submission`` (handles ``TVT`` vs ``tvt``).\\\"\\\"\\\"\\n    if target_name in train_df.columns:\\n        return train_df\\n    by_lower = {str(c).lower(): c for c in train_df.columns}\\n    key = str(target_name).lower()\\n    if key not in by_lower:\\n        raise ValueError(\\n            f\\\"Target {target_name!r} not in train columns and no case-insensitive match. \\\"\\n            f\\\"Train columns: {list(train_df.columns)}\\\"\\n        )\\n    actual = by_lower[key]\\n    out = train_df.rename(columns={actual: target_name})\\n    return out\\n\\n\\ndef build_feature_matrix(\\n    X: pd.DataFrame,\\n    *,\\n    numeric_cols: list[str],\\n    low_card_cols: list[str],\\n    high_card_cols: list[str],\\n) -> ColumnTransformer:\\n    \\\"\\\"\\\"Preprocessing: median imputation + one-hot / ordinal encoders.\\\"\\\"\\\"\\n    transformers: list[tuple[str, Any, list[str]]] = []\\n    if numeric_cols:\\n        transformers.append(\\n            (\\n                \\\"num\\\",\\n                Pipeline([(\\\"imputer\\\", SimpleImputer(strategy=\\\"median\\\"))]),\\n                numeric_cols,\\n            )\\n        )\\n    if low_card_cols:\\n        transformers.append(\\n            (\\n                \\\"cat_low\\\",\\n                Pipeline(\\n                    [\\n                        (\\\"imputer\\\", SimpleImputer(strategy=\\\"most_frequent\\\")),\\n                        (\\n                            \\\"oh\\\",\\n                            OneHotEncoder(handle_unknown=\\\"ignore\\\", sparse_output=False, max_categories=50),\\n                        ),\\n                    ]\\n                ),\\n                low_card_cols,\\n            )\\n        )\\n    if high_card_cols:\\n        transformers.append(\\n            (\\n                \\\"cat_high\\\",\\n                Pipeline(\\n                    [\\n                        (\\\"imputer\\\", SimpleImputer(strategy=\\\"most_frequent\\\")),\\n                        (\\\"ord\\\", OrdinalEncoder(handle_unknown=\\\"use_encoded_value\\\", unknown_value=-1)),\\n                    ]\\n                ),\\n                high_card_cols,\\n            )\\n        )\\n    if not transformers:\\n        raise ValueError(\\\"No feature columns after excluding id/target\\\")\\n    return ColumnTransformer(transformers=transformers, remainder=\\\"drop\\\", verbose_feature_names_out=False)\\n\\n\\ndef categorize_columns(\\n    df: pd.DataFrame,\\n    *,\\n    id_col: str,\\n    target_cols: list[str],\\n    max_card_for_onehot: int = 32,\\n) -> tuple[list[str], list[str], list[str]]:\\n    exclude = {id_col, *target_cols}\\n    numeric: list[str] = []\\n    low_card: list[str] = []\\n    high_card: list[str] = []\\n    for c in df.columns:\\n        if c in exclude:\\n            continue\\n        if pd.api.types.is_numeric_dtype(df[c]):\\n            numeric.append(c)\\n            continue\\n        nuniq = df[c].astype(str).nunique(dropna=False)\\n        if nuniq <= max_card_for_onehot:\\n            low_card.append(c)\\n        else:\\n            high_card.append(c)\\n    return numeric, low_card, high_card\\n\\n\\ndef _configure_preprocessor_output(preprocessor: ColumnTransformer) -> None:\\n    \\\"\\\"\\\"Prefer pandas transform output when the sklearn version supports it.\\\"\\\"\\\"\\n    try:\\n        preprocessor.set_output(transform=\\\"pandas\\\")\\n    except Exception:\\n        pass\\n\\n\\ndef _prepare_lgbm_features(preprocessor: ColumnTransformer, X: pd.DataFrame) -> pd.DataFrame:\\n    \\\"\\\"\\\"LightGBM sklearn API always records ``feature_names_in_``; use named columns.\\n\\n    Without this, ``fit`` on a bare ndarray still sets names like ``Column_0`` and\\n    ``predict`` on ndarray triggers sklearn's feature-name validation warning.\\n    \\\"\\\"\\\"\\n    Xt = preprocessor.transform(X)\\n    names = list(preprocessor.get_feature_names_out())\\n    if isinstance(Xt, pd.DataFrame):\\n        if list(Xt.columns) != names:\\n            Xt = Xt.set_axis(names, axis=1)\\n        return Xt\\n    return pd.DataFrame(Xt, columns=names, index=X.index)\\n\\n\\ndef make_estimator(n_rows: int):\\n    if _HAS_LGBM:\\n        return lgb.LGBMRegressor(\\n            objective=\\\"regression\\\",\\n            metric=\\\"rmse\\\",\\n            n_estimators=2000,\\n            learning_rate=0.05,\\n            num_leaves=63,\\n            subsample=0.85,\\n            colsample_bytree=0.85,\\n            reg_lambda=1.0,\\n            random_state=42,\\n            verbose=-1,\\n        )\\n    return HistGradientBoostingRegressor(\\n        max_depth=10,\\n        max_iter=500,\\n        learning_rate=0.06,\\n        random_state=42,\\n    )\\n\\n\\ndef _effective_cv(\\n    train_df: pd.DataFrame,\\n    test_df: pd.DataFrame,\\n    *,\\n    id_col: str,\\n    n_splits_req: int,\\n) -> tuple[str, np.ndarray | None, int]:\\n    \\\"\\\"\\\"Return ``(scheme, groups_or_none, n_splits)`` for CV.\\\"\\\"\\\"\\n    scheme = choose_cv_scheme(train_df, test_df, id_column=id_col)\\n    group_key = wgd.recommend_group_key(train_df, test_df, id_column=id_col)\\n    if scheme != \\\"groupkfold\\\" or group_key is None:\\n        return \\\"kfold\\\", None, n_splits_req\\n    groups = wgd.provide_groups(train_df, group_key, id_column=id_col)\\n    n_groups = wgd.count_unique_groups(groups)\\n    if n_groups < 2:\\n        return \\\"kfold\\\", None, n_splits_req\\n    n_splits_eff = min(n_splits_req, n_groups)\\n    if n_splits_eff < 2:\\n        return \\\"kfold\\\", None, min(n_splits_req, max(2, len(train_df) // 2))\\n    return \\\"groupkfold\\\", groups, n_splits_eff\\n\\n\\ndef cross_val_and_predict(\\n    X: pd.DataFrame,\\n    y_fit: np.ndarray,\\n    y_original: np.ndarray,\\n    X_test: pd.DataFrame,\\n    preprocessor: ColumnTransformer,\\n    *,\\n    scheme: str,\\n    groups: np.ndarray | None,\\n    n_splits: int,\\n    use_log1p: bool,\\n    val_row_mask: np.ndarray | None = None,\\n) -> tuple[float, list[float], np.ndarray, np.ndarray]:\\n    \\\"\\\"\\\"OOF RMSE on **original** scale, per-fold RMSEs, stacked test preds per fold, OOF predictions in fit space.\\\"\\\"\\\"\\n    fold_iter = emit_fold_indices(\\n        scheme if scheme == \\\"groupkfold\\\" else \\\"kfold\\\",\\n        X,\\n        n_splits=n_splits,\\n        groups=groups,\\n        shuffle=True,\\n        random_state=42,\\n    )\\n    oof_fit = np.full(len(X), np.nan, dtype=np.float64)\\n    test_blocks: list[np.ndarray] = []\\n    fold_rmses: list[float] = []\\n\\n    for train_idx, val_idx in fold_iter:\\n        if val_row_mask is not None and len(val_row_mask) == len(X):\\n            val_idx = val_idx[val_row_mask[val_idx]]\\n            if len(val_idx) == 0:\\n                continue\\n\\n        if scheme == \\\"groupkfold\\\" and groups is not None:\\n            g_tr = set(np.asarray(groups)[train_idx].astype(str))\\n            g_va = set(np.asarray(groups)[val_idx].astype(str))\\n            assert not (g_tr & g_va), \\\"GroupKFold leak: train and val share a group id\\\"\\n\\n        prep: ColumnTransformer = clone(preprocessor)\\n        if _HAS_LGBM:\\n            _configure_preprocessor_output(prep)\\n        X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]\\n        y_tr_fit = y_fit[train_idx]\\n        y_va_orig = y_original[val_idx]\\n        prep.fit(X_tr, y_tr_fit)\\n        if _HAS_LGBM:\\n            Xtr_t = _prepare_lgbm_features(prep, X_tr)\\n            Xva_t = _prepare_lgbm_features(prep, X_va)\\n            Xte_t = _prepare_lgbm_features(prep, X_test)\\n        else:\\n            Xtr_t = prep.transform(X_tr)\\n            Xva_t = prep.transform(X_va)\\n            Xte_t = prep.transform(X_test)\\n        est = make_estimator(len(X_tr))\\n        if _HAS_LGBM and isinstance(est, lgb.LGBMRegressor):\\n            est.fit(\\n                Xtr_t,\\n                y_tr_fit,\\n                eval_set=[(Xva_t, y_fit[val_idx])],\\n                callbacks=[lgb.early_stopping(50, verbose=False)],\\n            )\\n        else:\\n            est.fit(Xtr_t, y_tr_fit)\\n        pred_va_fit = est.predict(Xva_t)\\n        oof_fit[val_idx] = pred_va_fit\\n        if use_log1p:\\n            pred_va = np.expm1(np.clip(pred_va_fit, a_min=None, a_max=20.0))\\n            pred_va = np.clip(pred_va, a_min=0.0, a_max=None)\\n            fold_rmses.append(rmse(y_va_orig, pred_va))\\n        else:\\n            fold_rmses.append(rmse(y_va_orig, pred_va_fit))\\n        test_blocks.append(est.predict(Xte_t))\\n\\n    mean_rmse = float(np.mean(fold_rmses)) if fold_rmses else float(\\\"nan\\\")\\n    test_mat = np.column_stack(test_blocks) if test_blocks else np.zeros((len(X_test), 0))\\n    return mean_rmse, fold_rmses, test_mat, oof_fit\\n\\n\\ndef main() -> int:\\n    parser = argparse.ArgumentParser(description=\\\"ROGII Wellbore Geology \\u2014 RMSE baseline\\\")\\n    parser.add_argument(\\\"--data-dir\\\", type=Path, default=Path(\\\"data\\\"), help=\\\"Directory with train/test/sample_submission\\\")\\n    parser.add_argument(\\\"--out\\\", type=Path, default=Path(\\\"submission.csv\\\"))\\n    parser.add_argument(\\\"--n-splits\\\", type=int, default=5)\\n    parser.add_argument(\\\"--metrics-json\\\", type=Path, default=None, help=\\\"Optional path to write CV metrics JSON\\\")\\n    parser.add_argument(\\n        \\\"--artifacts-dir\\\",\\n        type=Path,\\n        default=None,\\n        help=\\\"If set, write transform.json and test_preds_per_fold.npy (notebook-compatible artifacts)\\\",\\n    )\\n    args = parser.parse_args()\\n    data_dir = args.data_dir.resolve()\\n    if not data_dir.is_dir():\\n        print(f\\\"data-dir not found: {data_dir}\\\", file=sys.stderr)\\n        return 1\\n\\n    try:\\n        train_path = _find_default_csv(data_dir, \\\"train\\\")\\n        test_path = _find_default_csv(data_dir, \\\"test\\\")\\n        sample_path = _find_default_csv(data_dir, \\\"sample_submission\\\")\\n    except FileNotFoundError as exc:\\n        print(str(exc), file=sys.stderr)\\n        return 1\\n\\n    train_df = pd.read_csv(train_path)\\n    test_df = pd.read_csv(test_path)\\n    sample_sub = pd.read_csv(sample_path, nrows=0)\\n    id_col, target_cols = infer_columns(sample_sub)\\n\\n    if len(target_cols) != 1:\\n        print(\\n            \\\"This baseline supports exactly one prediction column after the id. \\\"\\n            f\\\"Found: {target_cols}. Extend train_predict.py for multi-target.\\\",\\n            file=sys.stderr,\\n        )\\n        return 1\\n    target = target_cols[0]\\n    try:\\n        train_df = align_train_target_to_schema(train_df, target)\\n    except ValueError as exc:\\n        print(str(exc), file=sys.stderr)\\n        return 1\\n\\n    train_df = ensure_id_column(train_df, id_col)\\n    test_df = ensure_id_column(test_df, id_col)\\n\\n    y_raw = train_df[target].astype(np.float64).values\\n    log_rec = recommend_log1p(y_raw, skew_threshold=1.5)\\n    use_log1p = bool(log_rec[\\\"use_log1p\\\"])\\n    is_positive = bool(log_rec[\\\"strict_positivity\\\"][\\\"strict_positive\\\"])\\n    y_fit = np.log1p(np.clip(y_raw, a_min=0.0, a_max=None)) if use_log1p else y_raw\\n    y_original = y_raw\\n\\n    feature_cols = [c for c in train_df.columns if c not in (id_col, target) and c in test_df.columns]\\n    if not feature_cols:\\n        print(\\\"No overlapping feature columns between train and test (excluding id/target).\\\", file=sys.stderr)\\n        return 1\\n\\n    X_train = replace_sentinels_with_nan(train_df[feature_cols].copy(), sentinel_values=_SENTINELS)\\n    X_test = replace_sentinels_with_nan(test_df[feature_cols].copy(), sentinel_values=_SENTINELS)\\n\\n    num, low_c, high_c = categorize_columns(\\n        pd.concat([X_train, X_test], axis=0, ignore_index=True),\\n        id_col=id_col,\\n        target_cols=[target],\\n    )\\n\\n    preprocessor = build_feature_matrix(\\n        X_train,\\n        numeric_cols=num,\\n        low_card_cols=low_c,\\n        high_card_cols=high_c,\\n    )\\n    if _HAS_LGBM:\\n        _configure_preprocessor_output(preprocessor)\\n\\n    scheme, groups, n_splits = _effective_cv(train_df, test_df, id_col=id_col, n_splits_req=args.n_splits)\\n    group_key = wgd.recommend_group_key(train_df, test_df, id_column=id_col)\\n    print(f\\\"CV scheme: {scheme} (n_splits={n_splits})\\\" + (f\\\", group_key={group_key!r}\\\" if group_key else \\\"\\\"))\\n\\n    cv_rmse, fold_rmses, test_mat, oof_fit = cross_val_and_predict(\\n        X_train,\\n        y_fit,\\n        y_original,\\n        X_test,\\n        preprocessor,\\n        scheme=scheme,\\n        groups=groups,\\n        n_splits=n_splits,\\n        use_log1p=use_log1p,\\n    )\\n\\n    cv_rmse_transformed = (\\n        float(np.sqrt(np.nanmean((y_fit - oof_fit) ** 2))) if np.any(np.isfinite(oof_fit)) else None\\n    )\\n\\n    test_mean_fit = test_mat.mean(axis=1)\\n    if use_log1p:\\n        test_pred = np.expm1(np.clip(test_mean_fit, a_min=None, a_max=20.0))\\n        if is_positive:\\n            test_pred = np.clip(test_pred, a_min=0.0, a_max=None)\\n    else:\\n        test_pred = test_mean_fit\\n        if is_positive:\\n            test_pred = np.clip(test_pred, a_min=0.0, a_max=None)\\n\\n    print(f\\\"OOF RMSE (original target scale, mean over folds): {cv_rmse:.6f}\\\")\\n    if cv_rmse_transformed is not None and use_log1p:\\n        print(f\\\"OOF RMSE (log1p training scale): {cv_rmse_transformed:.6f}\\\")\\n    print(f\\\"use_log1p={use_log1p}; backend={'lightgbm' if _HAS_LGBM else 'sklearn HistGradientBoostingRegressor'}\\\")\\n\\n    out_df = pd.DataFrame({id_col: test_df[id_col], target: test_pred})\\n    out_df = out_df[[c for c in sample_sub.columns if c in out_df.columns]]\\n    args.out.parent.mkdir(parents=True, exist_ok=True)\\n    out_df.to_csv(args.out, index=False)\\n    print(f\\\"Wrote {args.out.resolve()} ({len(out_df)} rows)\\\")\\n\\n    if args.metrics_json:\\n        payload = {\\n            \\\"cv_rmse\\\": cv_rmse,\\n            \\\"cv_rmse_transformed\\\": cv_rmse_transformed,\\n            \\\"fold_rmses\\\": fold_rmses,\\n            \\\"n_folds\\\": n_splits,\\n            \\\"cv_scheme\\\": scheme,\\n            \\\"group_key\\\": group_key,\\n            \\\"backend\\\": \\\"lightgbm\\\" if _HAS_LGBM else \\\"sklearn.hist_gradient_boosting\\\",\\n            \\\"id_column\\\": id_col,\\n            \\\"target_column\\\": target,\\n            \\\"use_log1p\\\": use_log1p,\\n            \\\"is_positive\\\": is_positive,\\n        }\\n        args.metrics_json.write_text(json.dumps(payload, indent=2) + \\\"\\\\n\\\", encoding=\\\"utf-8\\\")\\n\\n    if args.artifacts_dir is not None:\\n        ad = args.artifacts_dir.resolve()\\n        ad.mkdir(parents=True, exist_ok=True)\\n        transform = {\\n            \\\"use_log1p\\\": use_log1p,\\n            \\\"is_positive\\\": is_positive,\\n            \\\"target_column\\\": target,\\n            \\\"id_column\\\": id_col,\\n            \\\"cv_rmse\\\": cv_rmse,\\n            \\\"cv_rmse_transformed\\\": cv_rmse_transformed,\\n            \\\"fold_rmses\\\": fold_rmses,\\n            \\\"group_col\\\": None if group_key is None or str(group_key).startswith(\\\"__derived\\\") else group_key,\\n            \\\"group_key_resolved\\\": group_key,\\n            \\\"n_folds\\\": n_splits,\\n            \\\"cv_scheme\\\": scheme,\\n            \\\"paths\\\": {\\n                \\\"train_csv\\\": str(train_path.resolve()),\\n                \\\"test_csv\\\": str(test_path.resolve()),\\n                \\\"sample_submission\\\": str(sample_path.resolve()),\\n            },\\n        }\\n        (ad / \\\"transform.json\\\").write_text(json.dumps(transform, indent=2) + \\\"\\\\n\\\", encoding=\\\"utf-8\\\")\\n        np.save(ad / \\\"test_preds_per_fold.npy\\\", test_mat)\\n        print(f\\\"Wrote {ad / 'transform.json'} and test_preds_per_fold.npy\\\")\\n\\n    try:\\n        from pipeline.tcn_runner import write_artifacts_index\\n\\n        write_artifacts_index(_ROOT)\\n    except Exception:\\n        pass\\n\\n    return 0\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    raise SystemExit(main())\\n\", \"pipeline/__init__.py\": \"\\\"\\\"\\\"ROGII competition ML pipeline building blocks (download \\u2192 EDA \\u2192 train \\u2192 submit).\\n\\nEach submodule exposes the functions named in the design grid (see submodule docstrings).\\nRun from the competition directory with ``PYTHONPATH`` including the repo root if you\\nuse ``data_analyst`` / ``trace_language`` imports.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\n__all__ = [\\n    \\\"data_downloader\\\",\\n    \\\"eda_profiler\\\",\\n    \\\"well_group_detector\\\",\\n    \\\"target_diagnostician\\\",\\n    \\\"feature_engineer\\\",\\n    \\\"preprocessor\\\",\\n    \\\"cv_orchestrator\\\",\\n    \\\"model_trainer\\\",\\n    \\\"model_ensembler\\\",\\n    \\\"predictor\\\",\\n    \\\"oof_evaluator\\\",\\n    \\\"error_analyzer\\\",\\n    \\\"submission_formatter\\\",\\n    \\\"submission_validator\\\",\\n    \\\"kaggle_submitter\\\",\\n]\\n\", \"pipeline/competition_data.py\": \"\\\"\\\"\\\"Load ROGII competition data from flat or multi-well Kaggle layouts.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport re\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\n\\n_WELL_RE = re.compile(r\\\"_?(?:train|test|)_?([0-9a-f]+)_\\\", re.I)\\n\\n\\ndef _well_from_stem(stem: str) -> str:\\n    m = _WELL_RE.match(stem + \\\"_\\\")\\n    if m:\\n        return m.group(1).lower()\\n    if \\\"__horizontal_well\\\" in stem:\\n        return stem.split(\\\"__\\\")[0].lstrip(\\\"_\\\")\\n    return stem\\n\\n\\ndef _glob_horizontal(dir_path: Path, split: str) -> list[Path]:\\n    if not dir_path.is_dir():\\n        return []\\n    return sorted(dir_path.glob(f\\\"*__horizontal_well.csv\\\")) or sorted(\\n        dir_path.glob(f\\\"_{split}_*horizontal_well.csv\\\")\\n    )\\n\\n\\ndef discover_layout(data_dir: Path) -> str:\\n    data_dir = data_dir.resolve()\\n    if (data_dir / \\\"train\\\").is_dir() and _glob_horizontal(data_dir / \\\"train\\\", \\\"train\\\"):\\n        return \\\"kaggle_multiwell\\\"\\n    if list(data_dir.glob(\\\"_train_*horizontal_well.csv\\\")):\\n        return \\\"flat\\\"\\n    # Single merged train.csv (must not match *train*.csv \\u2014 that glob hits train.csv)\\n    if (data_dir / \\\"train.csv\\\").is_file():\\n        return \\\"merged_csv\\\"\\n    return \\\"unknown\\\"\\n\\n\\ndef _read_well_csv(path: Path, *, well_id: str, is_train: bool) -> pd.DataFrame:\\n    df = pd.read_csv(path)\\n    df[\\\"well_id\\\"] = well_id\\n    df[\\\"is_train\\\"] = is_train\\n    return df\\n\\n\\ndef load_competition_frames(\\n    data_dir: Path,\\n) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, str, str]:\\n    \\\"\\\"\\\"Return ``train_df, test_df, sample_sub, id_col, target_col``.\\\"\\\"\\\"\\n    data_dir = data_dir.resolve()\\n    sample_path = data_dir / \\\"sample_submission.csv\\\"\\n    if not sample_path.is_file():\\n        raise FileNotFoundError(f\\\"Missing sample_submission.csv under {data_dir}\\\")\\n    sample_sub = pd.read_csv(sample_path)\\n    cols = list(sample_sub.columns)\\n    if len(cols) < 2:\\n        raise ValueError(\\\"sample_submission needs id + target columns\\\")\\n    id_col, target_col = cols[0], cols[1]\\n\\n    layout = discover_layout(data_dir)\\n    train_parts: list[pd.DataFrame] = []\\n    test_parts: list[pd.DataFrame] = []\\n\\n    if layout == \\\"kaggle_multiwell\\\":\\n        for p in _glob_horizontal(data_dir / \\\"train\\\", \\\"train\\\"):\\n            wid = _well_from_stem(p.stem)\\n            train_parts.append(_read_well_csv(p, well_id=wid, is_train=True))\\n        for p in _glob_horizontal(data_dir / \\\"test\\\", \\\"test\\\"):\\n            wid = _well_from_stem(p.stem)\\n            test_parts.append(_read_well_csv(p, well_id=wid, is_train=False))\\n    elif layout == \\\"flat\\\":\\n        for p in sorted(data_dir.glob(\\\"_train_*horizontal_well.csv\\\")):\\n            wid = _well_from_stem(p.stem)\\n            train_parts.append(_read_well_csv(p, well_id=wid, is_train=True))\\n        for p in sorted(data_dir.glob(\\\"_test_*horizontal_well.csv\\\")):\\n            wid = _well_from_stem(p.stem)\\n            test_parts.append(_read_well_csv(p, well_id=wid, is_train=False))\\n        if not train_parts:\\n            tp = data_dir / \\\"train.csv\\\"\\n            if tp.is_file():\\n                train_parts.append(_read_well_csv(tp, well_id=\\\"default\\\", is_train=True))\\n        if not test_parts:\\n            tp = data_dir / \\\"test.csv\\\"\\n            if tp.is_file():\\n                test_parts.append(_read_well_csv(tp, well_id=\\\"default\\\", is_train=False))\\n    elif layout == \\\"merged_csv\\\":\\n        train_parts.append(_read_well_csv(data_dir / \\\"train.csv\\\", well_id=\\\"default\\\", is_train=True))\\n        test_parts.append(_read_well_csv(data_dir / \\\"test.csv\\\", well_id=\\\"default\\\", is_train=False))\\n    else:\\n        raise FileNotFoundError(f\\\"Unrecognized data layout under {data_dir}\\\")\\n\\n    if not train_parts or not test_parts:\\n        raise FileNotFoundError(f\\\"Could not locate train/test CSVs in {data_dir} (layout={layout})\\\")\\n\\n    train_df = pd.concat(train_parts, ignore_index=True)\\n    test_df = pd.concat(test_parts, ignore_index=True)\\n\\n    # Align target column name (TVT vs tvt)\\n    if target_col not in train_df.columns:\\n        by_lower = {str(c).lower(): c for c in train_df.columns}\\n        if target_col.lower() in by_lower:\\n            train_df = train_df.rename(columns={by_lower[target_col.lower()]: target_col})\\n\\n    # Synthetic row ids when missing (dev smoke)\\n    if id_col not in test_df.columns:\\n        test_df[id_col] = (\\n            test_df[\\\"well_id\\\"].astype(str) + \\\"_\\\" + test_df.groupby(\\\"well_id\\\").cumcount().astype(str)\\n        )\\n    if id_col not in train_df.columns:\\n        train_df[id_col] = (\\n            train_df[\\\"well_id\\\"].astype(str) + \\\"_\\\" + train_df.groupby(\\\"well_id\\\").cumcount().astype(str)\\n        )\\n\\n    # Parse well + row index from submission ids for alignment\\n    sample_sub = sample_sub.copy()\\n    if \\\"well_id\\\" not in sample_sub.columns and \\\"_\\\" in str(sample_sub[id_col].iloc[0]):\\n        parts = sample_sub[id_col].astype(str).str.split(\\\"_\\\", n=1, expand=True)\\n        sample_sub[\\\"well_id\\\"] = parts[0]\\n        sample_sub[\\\"row_idx\\\"] = pd.to_numeric(parts[1], errors=\\\"coerce\\\").fillna(0).astype(int)\\n\\n    return train_df, test_df, sample_sub, id_col, target_col\\n\\n\\ndef load_typewell_lookup(data_dir: Path) -> dict[str, pd.DataFrame]:\\n    \\\"\\\"\\\"Map well_id \\u2192 typewell dataframe (may be empty for dev layout).\\\"\\\"\\\"\\n    data_dir = data_dir.resolve()\\n    out: dict[str, pd.DataFrame] = {}\\n    candidates = list(data_dir.glob(\\\"*__typewell.csv\\\"))\\n    candidates += list((data_dir / \\\"train\\\").glob(\\\"*__typewell.csv\\\")) if (data_dir / \\\"train\\\").is_dir() else []\\n    for p in candidates:\\n        # Skip macOS AppleDouble / hidden files (e.g. ._000d7d20__typewell.csv).\\n        if p.name.startswith(\\\"._\\\") or p.name.startswith(\\\".\\\"):\\n            continue\\n        wid = p.stem.replace(\\\"__typewell\\\", \\\"\\\").lstrip(\\\"_\\\")\\n        out[wid] = pd.read_csv(p)\\n    return out\\n\", \"pipeline/cv_orchestrator.py\": \"\\\"\\\"\\\"CV scheme selection and fold index emission.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Any, Iterator\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.model_selection import GroupKFold, KFold\\n\\nfrom pipeline import well_group_detector as wgd\\n\\n\\ndef choose_cv_scheme(\\n    train_df: pd.DataFrame,\\n    test_df: pd.DataFrame | None = None,\\n    *,\\n    id_column: str = \\\"id\\\",\\n) -> str:\\n    key = wgd.recommend_group_key(train_df, test_df, id_column=id_column)\\n    if key is None:\\n        return \\\"kfold\\\"\\n    groups = wgd.provide_groups(train_df, key, id_column=id_column)\\n    if wgd.count_unique_groups(groups) >= 2:\\n        return \\\"groupkfold\\\"\\n    return \\\"kfold\\\"\\n\\n\\ndef emit_fold_indices(\\n    scheme: str,\\n    X: pd.DataFrame,\\n    *,\\n    n_splits: int = 5,\\n    groups: np.ndarray | None = None,\\n    shuffle: bool = True,\\n    random_state: int = 42,\\n) -> Iterator[tuple[np.ndarray, np.ndarray]]:\\n    n = len(X)\\n    if scheme == \\\"groupkfold\\\" and groups is not None:\\n        n_groups = wgd.count_unique_groups(groups)\\n        n_eff = min(n_splits, n_groups)\\n        if n_eff >= 2:\\n            splitter = GroupKFold(n_splits=n_eff)\\n            yield from splitter.split(X, groups=groups)\\n            return\\n    n_eff = min(n_splits, max(2, n // 2))\\n    splitter = KFold(n_splits=n_eff, shuffle=shuffle, random_state=random_state)\\n    yield from splitter.split(X)\\n\\n\\ndef subdivide_train_by_well(train_df: pd.DataFrame, group_key: str, *, id_column: str = \\\"id\\\") -> dict:\\n    groups = wgd.provide_groups(train_df, group_key, id_column=id_column)\\n    wells = sorted(set(map(str, groups)))\\n    return {\\\"wells\\\": wells, \\\"n_wells\\\": len(wells)}\\n\\n\\ndef subdivide_train_by_depth_bin(train_df: pd.DataFrame, *, depth_col: str = \\\"MD\\\", q: int = 5) -> dict:\\n    if depth_col not in train_df.columns:\\n        return {\\\"bins\\\": [], \\\"note\\\": f\\\"column {depth_col!r} missing\\\"}\\n    bins = pd.qcut(train_df[depth_col], q=q, labels=False, duplicates=\\\"drop\\\")\\n    return {\\\"n_bins\\\": int(bins.nunique()), \\\"depth_col\\\": depth_col}\\n\\n\\ndef nested_groupkfold_emit_fold_indices(\\n    X: pd.DataFrame,\\n    *,\\n    groups: np.ndarray,\\n    n_splits: int = 5,\\n) -> list[dict[str, list[int]]]:\\n    folds: list[dict[str, list[int]]] = []\\n    for tr, va in emit_fold_indices(\\\"groupkfold\\\", X, n_splits=n_splits, groups=groups):\\n        folds.append({\\\"train_idx\\\": tr.tolist(), \\\"val_idx\\\": va.tolist()})\\n    return folds\\n\", \"pipeline/nb_support.py\": \"\\\"\\\"\\\"Shared path resolution for Rogii Jupyter notebooks (cwd-safe).\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport importlib.util\\nfrom pathlib import Path\\nfrom typing import Any\\n\\n\\ndef competition_root() -> Path:\\n    \\\"\\\"\\\"Directory that contains ``pipeline/`` and ``data/`` (Rogii competition folder).\\\"\\\"\\\"\\n    here = Path.cwd().resolve()\\n    for base in (here, *here.parents):\\n        if (base / \\\"pipeline\\\").is_dir() and (base / \\\"data\\\").is_dir():\\n            return base\\n    # Fallback: parent of notebooks/\\n    if (here / \\\"pipeline\\\").is_dir():\\n        return here\\n    return here.parent\\n\\n\\ndef load_train_predict():\\n    root = competition_root()\\n    path = root / \\\"train_predict.py\\\"\\n    spec = importlib.util.spec_from_file_location(\\\"rogii_train_predict_nb\\\", path)\\n    if spec is None or spec.loader is None:\\n        raise ImportError(f\\\"cannot load {path}\\\")\\n    mod = importlib.util.module_from_spec(spec)\\n    spec.loader.exec_module(mod)\\n    return mod\\n\\n\\ndef resolve_tabular_csvs(data_dir: Path | None = None) -> tuple[Path, Path, Path]:\\n    \\\"\\\"\\\"Return ``(train_path, test_path, sample_submission_path)`` using ``train_predict`` heuristics.\\\"\\\"\\\"\\n    tp = load_train_predict()\\n    d = (data_dir or competition_root() / \\\"data\\\").resolve()\\n    return (\\n        tp._find_default_csv(d, \\\"train\\\"),\\n        tp._find_default_csv(d, \\\"test\\\"),\\n        tp._find_default_csv(d, \\\"sample_submission\\\"),\\n    )\\n\\n\\ndef ensure_id_column(df: Any, id_col: str) -> Any:\\n    \\\"\\\"\\\"If ``id_col`` is missing, add a synthetic id for offline notebook smoke (not for Kaggle).\\\"\\\"\\\"\\n    import numpy as np\\n\\n    if id_col in df.columns:\\n        return df\\n    out = df.copy()\\n    out.insert(0, id_col, np.arange(len(out), dtype=np.int64).astype(str))\\n    return out\\n\\n\\ndef build_aligned_sample_submission(\\n    sample_sub: Any,\\n    test_df: Any,\\n    *,\\n    id_col: str,\\n    target_col: str,\\n    out_path: Path,\\n) -> Path:\\n    \\\"\\\"\\\"Build a ``sample_submission``-shaped frame with one row per test row (for envelope validator).\\\"\\\"\\\"\\n    import pandas as pd\\n\\n    aligned = pd.DataFrame({id_col: test_df[id_col].values})\\n    for c in sample_sub.columns:\\n        if c == id_col:\\n            continue\\n        aligned[c] = 0.0 if c == target_col else 0.0\\n    out_path.parent.mkdir(parents=True, exist_ok=True)\\n    aligned.to_csv(out_path, index=False)\\n    return out_path\\n\", \"pipeline/preprocessor.py\": \"\\\"\\\"\\\"Preprocessor helpers for Rogii baseline (Pedregosa ColumnTransformer path).\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Any\\n\\nimport pandas as pd\\n\\n\\ndef replace_sentinels_with_nan(\\n    df: pd.DataFrame,\\n    *,\\n    sentinel_values: tuple[float, ...] = (-999, -999.25, -9999, -99999),\\n) -> pd.DataFrame:\\n    out = df.copy()\\n    for c in out.columns:\\n        if pd.api.types.is_numeric_dtype(out[c]):\\n            out[c] = out[c].replace(list(sentinel_values), pd.NA)\\n    return out\\n\", \"pipeline/target_diagnostician.py\": \"\\\"\\\"\\\"Target diagnostics for TVT (log1p recommendation).\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport numpy as np\\n\\n\\ndef _skewness(y: np.ndarray) -> float:\\n    y = np.asarray(y, dtype=np.float64)\\n    y = y[np.isfinite(y)]\\n    if len(y) < 3:\\n        return 0.0\\n    m = y.mean()\\n    sd = y.std()\\n    if sd == 0:\\n        return 0.0\\n    return float(np.mean(((y - m) / sd) ** 3))\\n\\n\\ndef recommend_log1p(y: np.ndarray, *, skew_threshold: float = 1.5) -> dict:\\n    y = np.asarray(y, dtype=np.float64)\\n    finite = y[np.isfinite(y)]\\n    strict_positive = bool(len(finite) > 0 and np.all(finite > 0))\\n    skew = _skewness(finite)\\n    use = strict_positive and skew > skew_threshold\\n    return {\\n        \\\"use_log1p\\\": use,\\n        \\\"skewness\\\": skew,\\n        \\\"skew_threshold\\\": skew_threshold,\\n        \\\"strict_positivity\\\": {\\\"strict_positive\\\": strict_positive},\\n    }\\n\", \"pipeline/well_group_detector.py\": \"\\\"\\\"\\\"Well / group detection for leak-safe GroupKFold.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Any\\n\\nimport numpy as np\\nimport pandas as pd\\n\\n_WELL_HINTS = (\\\"well\\\", \\\"hole\\\", \\\"pad\\\", \\\"well_id\\\", \\\"wellid\\\", \\\"api\\\")\\n\\n\\ndef scan_for_well_columns(df: pd.DataFrame) -> list[str]:\\n    out: list[str] = []\\n    for c in df.columns:\\n        cl = str(c).lower()\\n        if any(h in cl for h in _WELL_HINTS):\\n            out.append(c)\\n    return out\\n\\n\\ndef recommend_group_key(\\n    train_df: pd.DataFrame,\\n    test_df: pd.DataFrame | None = None,\\n    *,\\n    id_column: str = \\\"id\\\",\\n) -> str | None:\\n    frames: list[pd.DataFrame] = [train_df]\\n    if test_df is not None:\\n        frames.append(test_df)\\n    for df in frames:\\n        for c in scan_for_well_columns(df):\\n            if df[c].nunique(dropna=False) >= 1:\\n                return c\\n    if id_column in train_df.columns:\\n        sample = train_df[id_column].astype(str).head(100)\\n        if sample.str.contains(\\\"_\\\").any():\\n            return f\\\"__derived_well_from_{id_column}\\\"\\n    return None\\n\\n\\ndef provide_groups(df: pd.DataFrame, group_key: str, *, id_column: str = \\\"id\\\") -> np.ndarray:\\n    if group_key.startswith(\\\"__derived_well_from_\\\"):\\n        col = group_key.replace(\\\"__derived_well_from_\\\", \\\"\\\", 1)\\n        if col not in df.columns:\\n            raise ValueError(f\\\"derived group column {col!r} missing\\\")\\n        return df[col].astype(str).str.split(\\\"_\\\").str[0].values\\n    if group_key not in df.columns:\\n        raise ValueError(f\\\"group_key {group_key!r} not in dataframe\\\")\\n    return df[group_key].astype(str).values\\n\\n\\ndef count_unique_groups(groups: np.ndarray) -> int:\\n    return int(pd.Series(groups).nunique(dropna=False))\\n\\n\\ndef check_train_test_overlap(\\n    train_groups: np.ndarray,\\n    test_groups: np.ndarray,\\n) -> dict[str, Any]:\\n    tr = set(map(str, train_groups))\\n    te = set(map(str, test_groups))\\n    overlap = tr & te\\n    return {\\\"n_train_groups\\\": len(tr), \\\"n_test_groups\\\": len(te), \\\"overlap_count\\\": len(overlap)}\\n\", \"pipeline/temporal_cnn.py\": \"\\\"\\\"\\\"Temporal CNN baseline for per-depth regression on wellbore data.\\n\\nArchitecture\\n------------\\nNon-causal (bidirectional) 1D temporal convolutional network with dilated\\nconvolutions and residual connections, after Bai et al. (\\\"An Empirical\\nEvaluation of Generic Convolutional and Recurrent Networks for Sequence\\nModeling\\\", 2018), adapted to a sequence-to-sequence regression head.\\n\\nThe wellbore data is intrinsically a *depth series* (rows ordered by\\nmeasured depth within a well), so we:\\n\\n1. Group rows by ``well_id``.\\n2. Sort each well by ``MD`` (measured depth).\\n3. Slice each well into overlapping fixed-length windows.\\n4. Train a per-position regressor on the window.\\n5. Reassemble overlapping window predictions back into a single per-row\\n   prediction by averaging the overlaps.\\n\\nThe module is GPU-aware but works fine on CPU for the 5 k-row-per-well\\nscale of this competition.\\n\\nAPI surface\\n-----------\\n- ``TemporalCNN`` \\u2014 the ``nn.Module``.\\n- ``make_sequences`` \\u2014 DataFrame \\u2192 windowed tensors + reassembly index.\\n- ``reassemble`` \\u2014 per-window predictions \\u2192 per-row predictions.\\n- ``train_one_fold`` \\u2014 single-fold training loop with early stopping.\\n- ``predict_windows`` \\u2014 batched inference helper.\\n- ``rmse`` \\u2014 root mean squared error (matches the leaderboard metric).\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass\\nfrom typing import Iterable, Sequence\\n\\nimport numpy as np\\nimport pandas as pd\\n\\ntry:\\n    import torch\\n    from torch import nn\\n    from torch.utils.data import DataLoader, TensorDataset\\nexcept ImportError as exc:  # pragma: no cover\\n    raise ImportError(\\n        \\\"pipeline.temporal_cnn requires torch. Install via \\\"\\n        \\\"`pip install torch --index-url https://download.pytorch.org/whl/cpu`\\\"\\n    ) from exc\\n\\n\\n# ----------------------------------------------------------------------\\n# Model\\n# ----------------------------------------------------------------------\\n\\n\\nclass _TCNBlock(nn.Module):\\n    \\\"\\\"\\\"Two-conv residual block with symmetric (non-causal) padding.\\\"\\\"\\\"\\n\\n    def __init__(\\n        self,\\n        in_ch: int,\\n        out_ch: int,\\n        kernel_size: int = 3,\\n        dilation: int = 1,\\n        dropout: float = 0.1,\\n    ) -> None:\\n        super().__init__()\\n        pad = ((kernel_size - 1) * dilation) // 2\\n        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, padding=pad, dilation=dilation)\\n        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, padding=pad, dilation=dilation)\\n        self.bn1 = nn.BatchNorm1d(out_ch)\\n        self.bn2 = nn.BatchNorm1d(out_ch)\\n        self.dropout = nn.Dropout(dropout)\\n        self.skip = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        out = torch.relu(self.bn1(self.conv1(x)))\\n        out = self.dropout(out)\\n        out = torch.relu(self.bn2(self.conv2(out)))\\n        out = self.dropout(out)\\n        return torch.relu(out + self.skip(x))\\n\\n\\nclass TemporalCNN(nn.Module):\\n    \\\"\\\"\\\"Stacked dilated 1D-CNN producing one prediction per depth tick.\\n\\n    Receptive field for the default config (4 blocks, kernel=3) is roughly\\n    ``2 * (kernel - 1) * (2^n_blocks - 1) + 1 = 2 * 2 * 15 + 1 = 61``\\n    depth ticks centered on the prediction position.\\n    \\\"\\\"\\\"\\n\\n    def __init__(\\n        self,\\n        n_features: int,\\n        hidden: int = 64,\\n        n_blocks: int = 4,\\n        kernel_size: int = 3,\\n        dropout: float = 0.1,\\n    ) -> None:\\n        super().__init__()\\n        layers = []\\n        ch_in = n_features\\n        for i in range(n_blocks):\\n            layers.append(_TCNBlock(ch_in, hidden, kernel_size, 2**i, dropout))\\n            ch_in = hidden\\n        self.blocks = nn.Sequential(*layers)\\n        self.head = nn.Conv1d(hidden, 1, kernel_size=1)\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        \\\"\\\"\\\"``x`` shape: ``(batch, n_features, window_len)``.\\n\\n        Returns ``(batch, window_len)``.\\n        \\\"\\\"\\\"\\n        h = self.blocks(x)\\n        return self.head(h).squeeze(1)\\n\\n\\n# ----------------------------------------------------------------------\\n# Sequence assembly\\n# ----------------------------------------------------------------------\\n\\n\\n@dataclass\\nclass SequenceMap:\\n    \\\"\\\"\\\"Bookkeeping needed to reassemble window predictions back to row order.\\n\\n    Attributes\\n    ----------\\n    row_indices\\n        For each window, the original DataFrame row indices it covers.\\n        Shape: ``(n_windows, window_len)``.\\n    n_rows\\n        Total number of rows in the source DataFrame (so a result array of\\n        length ``n_rows`` can be allocated for reassembly).\\n    \\\"\\\"\\\"\\n\\n    row_indices: np.ndarray\\n    n_rows: int\\n\\n\\ndef make_sequences(\\n    df: pd.DataFrame,\\n    *,\\n    well_col: str,\\n    depth_col: str,\\n    feature_cols: Sequence[str],\\n    target_col: str | None = None,\\n    window_len: int = 128,\\n    stride: int = 64,\\n    pad_value: float = 0.0,\\n) -> tuple[np.ndarray, np.ndarray | None, SequenceMap]:\\n    \\\"\\\"\\\"Build overlapping windows of length ``window_len`` per well.\\n\\n    The DataFrame is left intact; the returned ``SequenceMap.row_indices``\\n    array is keyed on the original DataFrame index after ``reset_index(drop=True)``\\n    has been applied to ``df`` prior to calling this function.\\n\\n    Returns\\n    -------\\n    X\\n        ``(n_windows, n_features, window_len)`` float32.\\n    y\\n        ``(n_windows, window_len)`` float32, or ``None`` if ``target_col`` is None.\\n        Padded positions carry ``pad_value`` (and are masked out at loss-time;\\n        see :func:`train_one_fold`).\\n    seq_map\\n        :class:`SequenceMap` for reassembly.\\n    \\\"\\\"\\\"\\n    if window_len <= 0 or stride <= 0:\\n        raise ValueError(\\\"window_len and stride must be positive\\\")\\n\\n    df = df.reset_index(drop=True)\\n    if well_col not in df.columns:\\n        raise ValueError(f\\\"well_col {well_col!r} not in dataframe\\\")\\n    if depth_col not in df.columns:\\n        raise ValueError(f\\\"depth_col {depth_col!r} not in dataframe\\\")\\n\\n    X_windows: list[np.ndarray] = []\\n    y_windows: list[np.ndarray] = []\\n    row_idx_windows: list[np.ndarray] = []\\n\\n    n_features = len(feature_cols)\\n    for _, well_df in df.groupby(well_col, sort=False):\\n        well_df = well_df.sort_values(depth_col, kind=\\\"mergesort\\\")\\n        rows = well_df.index.to_numpy()\\n        n = len(rows)\\n        if n == 0:\\n            continue\\n        feat = well_df[list(feature_cols)].to_numpy(dtype=np.float32, copy=True)\\n        feat = np.nan_to_num(feat, nan=0.0, posinf=0.0, neginf=0.0)\\n        if target_col is not None:\\n            tgt = well_df[target_col].to_numpy(dtype=np.float32, copy=True)\\n        else:\\n            tgt = None\\n\\n        # slide windows; pad the tail of the last window if needed\\n        starts = list(range(0, max(n - window_len, 0) + 1, stride))\\n        if not starts or starts[-1] + window_len < n:\\n            starts.append(max(n - window_len, 0))\\n        for s in starts:\\n            e = s + window_len\\n            x_block = np.full((window_len, n_features), pad_value, dtype=np.float32)\\n            r_block = np.full(window_len, -1, dtype=np.int64)\\n            actual = min(e, n) - s\\n            x_block[:actual] = feat[s : s + actual]\\n            r_block[:actual] = rows[s : s + actual]\\n            X_windows.append(x_block.T)  # (n_features, window_len)\\n            row_idx_windows.append(r_block)\\n            if tgt is not None:\\n                y_block = np.full(window_len, pad_value, dtype=np.float32)\\n                y_block[:actual] = tgt[s : s + actual]\\n                y_windows.append(y_block)\\n\\n    if not X_windows:\\n        raise ValueError(\\\"no windows produced \\u2014 check well_col / depth_col\\\")\\n\\n    X = np.stack(X_windows, axis=0)\\n    seq_map = SequenceMap(\\n        row_indices=np.stack(row_idx_windows, axis=0),\\n        n_rows=len(df),\\n    )\\n    y = np.stack(y_windows, axis=0) if y_windows else None\\n    return X, y, seq_map\\n\\n\\ndef reassemble(window_preds: np.ndarray, seq_map: SequenceMap) -> np.ndarray:\\n    \\\"\\\"\\\"Average overlapping window predictions back to per-row predictions.\\n\\n    Returns a ``(seq_map.n_rows,)`` array. Rows not covered by any window\\n    (shouldn't happen if sequences cover every well) are returned as NaN.\\n    \\\"\\\"\\\"\\n    if window_preds.shape != seq_map.row_indices.shape:\\n        raise ValueError(\\n            f\\\"shape mismatch: preds {window_preds.shape} vs \\\"\\n            f\\\"row_indices {seq_map.row_indices.shape}\\\"\\n        )\\n    sums = np.zeros(seq_map.n_rows, dtype=np.float64)\\n    counts = np.zeros(seq_map.n_rows, dtype=np.int64)\\n    flat_preds = window_preds.reshape(-1).astype(np.float64)\\n    flat_rows = seq_map.row_indices.reshape(-1)\\n    valid = flat_rows >= 0\\n    np.add.at(sums, flat_rows[valid], flat_preds[valid])\\n    np.add.at(counts, flat_rows[valid], 1)\\n    out = np.full(seq_map.n_rows, np.nan, dtype=np.float64)\\n    nz = counts > 0\\n    out[nz] = sums[nz] / counts[nz]\\n    return out\\n\\n\\n# ----------------------------------------------------------------------\\n# Training / inference\\n# ----------------------------------------------------------------------\\n\\n\\ndef rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:\\n    \\\"\\\"\\\"RMSE on the original target scale (Kaggle definition).\\\"\\\"\\\"\\n    y_true = np.asarray(y_true, dtype=np.float64).ravel()\\n    y_pred = np.asarray(y_pred, dtype=np.float64).ravel()\\n    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))\\n\\n\\ndef _make_loader(X: np.ndarray, y: np.ndarray | None, batch_size: int, shuffle: bool) -> DataLoader:\\n    Xt = torch.from_numpy(X).float()\\n    if y is None:\\n        ds = TensorDataset(Xt)\\n    else:\\n        ds = TensorDataset(Xt, torch.from_numpy(y).float())\\n    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False)\\n\\n\\ndef train_one_fold(\\n    X_tr: np.ndarray,\\n    y_tr: np.ndarray,\\n    X_va: np.ndarray,\\n    y_va: np.ndarray,\\n    *,\\n    n_features: int,\\n    hidden: int = 64,\\n    n_blocks: int = 4,\\n    kernel_size: int = 3,\\n    dropout: float = 0.1,\\n    lr: float = 1e-3,\\n    weight_decay: float = 1e-5,\\n    batch_size: int = 32,\\n    max_epochs: int = 60,\\n    patience: int = 8,\\n    pad_value: float = 0.0,\\n    device: str | None = None,\\n    verbose: bool = False,\\n) -> tuple[TemporalCNN, float, list[float]]:\\n    \\\"\\\"\\\"Train one fold; return (best_model, best_val_rmse, history).\\n\\n    The validation RMSE is computed on the **non-padded** positions only\\n    (positions whose corresponding row index is ``-1`` are masked out).\\n    \\\"\\\"\\\"\\n    dev = torch.device(device or (\\\"cuda\\\" if torch.cuda.is_available() else \\\"cpu\\\"))\\n    model = TemporalCNN(n_features, hidden=hidden, n_blocks=n_blocks,\\n                        kernel_size=kernel_size, dropout=dropout).to(dev)\\n    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)\\n\\n    train_loader = _make_loader(X_tr, y_tr, batch_size, shuffle=True)\\n    val_loader = _make_loader(X_va, y_va, batch_size, shuffle=False)\\n\\n    best_rmse = float(\\\"inf\\\")\\n    best_state: dict | None = None\\n    history: list[float] = []\\n    bad_epochs = 0\\n\\n    for epoch in range(max_epochs):\\n        model.train()\\n        train_loss = 0.0\\n        n_train = 0\\n        for xb, yb in train_loader:\\n            xb, yb = xb.to(dev), yb.to(dev)\\n            mask = (yb != pad_value).float()\\n            opt.zero_grad()\\n            pred = model(xb)\\n            sq = (pred - yb) ** 2 * mask\\n            loss = sq.sum() / mask.sum().clamp(min=1.0)\\n            loss.backward()\\n            opt.step()\\n            train_loss += float(loss.item()) * xb.size(0)\\n            n_train += xb.size(0)\\n        train_loss /= max(n_train, 1)\\n\\n        model.eval()\\n        sq_sum = 0.0\\n        m_sum = 0.0\\n        with torch.no_grad():\\n            for xb, yb in val_loader:\\n                xb, yb = xb.to(dev), yb.to(dev)\\n                mask = (yb != pad_value).float()\\n                pred = model(xb)\\n                sq_sum += float(((pred - yb) ** 2 * mask).sum().item())\\n                m_sum += float(mask.sum().item())\\n        val_rmse = float(np.sqrt(sq_sum / max(m_sum, 1.0)))\\n        history.append(val_rmse)\\n        if verbose:\\n            print(f\\\"  epoch {epoch:02d}  train_mse={train_loss:.6f}  val_rmse={val_rmse:.6f}\\\")\\n\\n        if val_rmse + 1e-9 < best_rmse:\\n            best_rmse = val_rmse\\n            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\\n            bad_epochs = 0\\n        else:\\n            bad_epochs += 1\\n            if bad_epochs >= patience:\\n                if verbose:\\n                    print(f\\\"  early stop @ epoch {epoch}\\\")\\n                break\\n\\n    if best_state is not None:\\n        model.load_state_dict(best_state)\\n    return model, best_rmse, history\\n\\n\\ndef predict_windows(\\n    model: TemporalCNN,\\n    X: np.ndarray,\\n    *,\\n    batch_size: int = 64,\\n    device: str | None = None,\\n) -> np.ndarray:\\n    \\\"\\\"\\\"Run the model in eval mode; return ``(n_windows, window_len)``.\\\"\\\"\\\"\\n    dev = torch.device(device or (\\\"cuda\\\" if torch.cuda.is_available() else \\\"cpu\\\"))\\n    model = model.to(dev).eval()\\n    loader = _make_loader(X, None, batch_size, shuffle=False)\\n    out_chunks: list[np.ndarray] = []\\n    with torch.no_grad():\\n        for (xb,) in loader:\\n            xb = xb.to(dev)\\n            preds = model(xb).cpu().numpy()\\n            out_chunks.append(preds)\\n    return np.concatenate(out_chunks, axis=0)\\n\", \"pipeline/episodic_benchmark.py\": \"\\\"\\\"\\\"Episodic TCN benchmark: track best checkpoint per fold and cross-episode performance.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nfrom dataclasses import asdict, dataclass, field\\nfrom datetime import datetime, timezone\\nfrom pathlib import Path\\nfrom typing import Any\\n\\n\\n@dataclass\\nclass EpisodeRecord:\\n    fold: int\\n    episode: int\\n    seed: int\\n    val_rmse: float\\n    best_window_rmse: float\\n    checkpoint: str\\n    n_epochs: int\\n    is_best_fold: bool = False\\n\\n\\n@dataclass\\nclass EpisodicBenchmark:\\n    \\\"\\\"\\\"Accumulates episodic training outcomes for benchmark reporting.\\\"\\\"\\\"\\n\\n    variant: str\\n    approach: str = \\\"\\\"\\n    paper_id: str = \\\"\\\"\\n    eval_mask: str = \\\"full_well\\\"\\n    use_log1p: bool = False\\n    feature_cols: list[str] = field(default_factory=list)\\n    cv_scheme: str = \\\"\\\"\\n    episodes_per_fold: int = 3\\n    max_epochs: int = 60\\n    patience: int = 8\\n    episodes: list[EpisodeRecord] = field(default_factory=list)\\n    fold_best: list[dict[str, Any]] = field(default_factory=list)\\n    oof_rmse: float | None = None\\n    oof_rmse_raw_scale: float | None = None\\n    fold_rmses: list[float] = field(default_factory=list)\\n    elapsed_seconds: float | None = None\\n\\n    def record_episode(\\n        self,\\n        *,\\n        fold: int,\\n        episode: int,\\n        seed: int,\\n        val_rmse: float,\\n        best_window_rmse: float,\\n        checkpoint: str,\\n        n_epochs: int,\\n    ) -> None:\\n        self.episodes.append(\\n            EpisodeRecord(\\n                fold=fold,\\n                episode=episode,\\n                seed=seed,\\n                val_rmse=val_rmse,\\n                best_window_rmse=best_window_rmse,\\n                checkpoint=checkpoint,\\n                n_epochs=n_epochs,\\n            )\\n        )\\n\\n    def set_fold_best(self, fold: int, checkpoint: str, val_rmse: float) -> None:\\n        self.fold_best.append({\\\"fold\\\": fold, \\\"checkpoint\\\": checkpoint, \\\"val_rmse\\\": val_rmse})\\n        for ep in self.episodes:\\n            if ep.fold == fold:\\n                ep.is_best_fold = ep.checkpoint == checkpoint\\n\\n    def finalize(\\n        self,\\n        *,\\n        oof_rmse: float,\\n        fold_rmses: list[float],\\n        elapsed_seconds: float,\\n        oof_rmse_raw_scale: float | None = None,\\n    ) -> dict[str, Any]:\\n        self.oof_rmse = oof_rmse\\n        self.oof_rmse_raw_scale = oof_rmse_raw_scale\\n        self.fold_rmses = fold_rmses\\n        self.elapsed_seconds = elapsed_seconds\\n        best_ep = min(self.episodes, key=lambda e: e.val_rmse) if self.episodes else None\\n        return {\\n            \\\"variant\\\": self.variant,\\n            \\\"approach\\\": self.approach,\\n            \\\"paper_id\\\": self.paper_id,\\n            \\\"eval_mask\\\": self.eval_mask,\\n            \\\"use_log1p\\\": self.use_log1p,\\n            \\\"n_features\\\": len(self.feature_cols),\\n            \\\"feature_cols\\\": self.feature_cols,\\n            \\\"cv_scheme\\\": self.cv_scheme,\\n            \\\"episodes_per_fold\\\": self.episodes_per_fold,\\n            \\\"max_epochs\\\": self.max_epochs,\\n            \\\"patience\\\": self.patience,\\n            \\\"oof_rmse\\\": self.oof_rmse,\\n            \\\"oof_rmse_raw_scale\\\": self.oof_rmse_raw_scale,\\n            \\\"mean_fold_rmse\\\": float(sum(fold_rmses) / len(fold_rmses)) if fold_rmses else None,\\n            \\\"std_fold_rmse\\\": float(\\n                (sum((x - sum(fold_rmses) / len(fold_rmses)) ** 2 for x in fold_rmses) / len(fold_rmses)) ** 0.5\\n            )\\n            if len(fold_rmses) > 1\\n            else 0.0,\\n            \\\"fold_rmses\\\": fold_rmses,\\n            \\\"fold_best\\\": self.fold_best,\\n            \\\"best_episode_overall\\\": asdict(best_ep) if best_ep else None,\\n            \\\"episodes\\\": [asdict(e) for e in self.episodes],\\n            \\\"elapsed_seconds\\\": elapsed_seconds,\\n            \\\"recorded_at\\\": datetime.now(timezone.utc).isoformat(),\\n        }\\n\\n    def write_json(self, path: Path) -> Path:\\n        payload = self.finalize(\\n            oof_rmse=self.oof_rmse or float(\\\"nan\\\"),\\n            fold_rmses=self.fold_rmses,\\n            elapsed_seconds=self.elapsed_seconds or 0.0,\\n            oof_rmse_raw_scale=self.oof_rmse_raw_scale,\\n        )\\n        path.parent.mkdir(parents=True, exist_ok=True)\\n        path.write_text(json.dumps(payload, indent=2), encoding=\\\"utf-8\\\")\\n        return path\\n\\n\\ndef load_benchmark(path: Path) -> dict[str, Any]:\\n    return json.loads(path.read_text(encoding=\\\"utf-8\\\"))\\n\\n\\ndef compare_variants(benchmark_paths: list[Path]) -> list[dict[str, Any]]:\\n    rows: list[dict[str, Any]] = []\\n    for p in sorted(benchmark_paths):\\n        if not p.is_file():\\n            continue\\n        b = load_benchmark(p)\\n        rows.append(\\n            {\\n                \\\"variant\\\": b.get(\\\"variant\\\"),\\n                \\\"oof_rmse\\\": b.get(\\\"oof_rmse\\\"),\\n                \\\"oof_rmse_raw_scale\\\": b.get(\\\"oof_rmse_raw_scale\\\"),\\n                \\\"mean_fold_rmse\\\": b.get(\\\"mean_fold_rmse\\\"),\\n                \\\"n_features\\\": b.get(\\\"n_features\\\"),\\n                \\\"eval_mask\\\": b.get(\\\"eval_mask\\\"),\\n                \\\"best_episode_val_rmse\\\": (b.get(\\\"best_episode_overall\\\") or {}).get(\\\"val_rmse\\\"),\\n                \\\"path\\\": str(p),\\n            }\\n        )\\n    return rows\\n\", \"pipeline/ensemble_blend.py\": \"\\\"\\\"\\\"OOF blending utilities (hill-climbing + Ridge meta-learner).\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport numpy as np\\nfrom sklearn.linear_model import Ridge\\nfrom sklearn.metrics import mean_squared_error\\n\\n\\ndef rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:\\n    y_true = np.asarray(y_true, dtype=np.float64).ravel()\\n    y_pred = np.asarray(y_pred, dtype=np.float64).ravel()\\n    mask = np.isfinite(y_true) & np.isfinite(y_pred)\\n    if not mask.any():\\n        return float(\\\"nan\\\")\\n    return float(np.sqrt(mean_squared_error(y_true[mask], y_pred[mask])))\\n\\n\\ndef _normalize_weights(w: np.ndarray) -> np.ndarray:\\n    w = np.clip(w, 0.0, None)\\n    s = w.sum()\\n    return w / s if s > 0 else np.ones_like(w) / len(w)\\n\\n\\ndef coordinate_descent_blend(\\n    oof_matrix: np.ndarray,\\n    y_true: np.ndarray,\\n    *,\\n    n_iter: int = 400,\\n    step: float = 0.02,\\n    seed: int = 42,\\n) -> tuple[np.ndarray, float]:\\n    \\\"\\\"\\\"Hill-climbing style weight search (karnak/Raunak pattern, no external wheel).\\\"\\\"\\\"\\n    rng = np.random.default_rng(seed)\\n    n_models = oof_matrix.shape[1]\\n    w = np.ones(n_models, dtype=np.float64) / n_models\\n    best_rmse = rmse(y_true, oof_matrix @ w)\\n\\n    for _ in range(n_iter):\\n        improved = False\\n        for j in range(n_models):\\n            for delta in (-step, step):\\n                trial = w.copy()\\n                trial[j] = max(0.0, trial[j] + delta)\\n                trial = _normalize_weights(trial)\\n                score = rmse(y_true, oof_matrix @ trial)\\n                if score < best_rmse - 1e-9:\\n                    best_rmse = score\\n                    w = trial\\n                    improved = True\\n        if not improved:\\n            # random jitter restart\\n            jitter = rng.dirichlet(np.ones(n_models))\\n            score = rmse(y_true, oof_matrix @ jitter)\\n            if score < best_rmse:\\n                best_rmse = score\\n                w = jitter\\n\\n    return w, best_rmse\\n\\n\\ndef ridge_stack_oof(\\n    oof_matrix: np.ndarray,\\n    y_true: np.ndarray,\\n    *,\\n    alpha: float = 1.0,\\n) -> tuple[np.ndarray, Ridge, float]:\\n    \\\"\\\"\\\"Fit Ridge meta-learner on OOF predictions.\\\"\\\"\\\"\\n    mask = np.all(np.isfinite(oof_matrix), axis=1) & np.isfinite(y_true)\\n    X = oof_matrix[mask]\\n    y = y_true[mask]\\n    model = Ridge(alpha=alpha, positive=True, fit_intercept=True)\\n    model.fit(X, y)\\n    pred = model.predict(oof_matrix)\\n    return pred, model, rmse(y_true, pred)\\n\\n\\ndef apply_savgol_safe(y: np.ndarray, *, window: int = 17, poly: int = 3) -> np.ndarray:\\n    \\\"\\\"\\\"Per-well Savgol smoothing when scipy available; else return input.\\\"\\\"\\\"\\n    try:\\n        from scipy.signal import savgol_filter\\n    except ImportError:\\n        return y\\n    w = window if window % 2 == 1 else window + 1\\n    w = min(w, max(5, len(y) - (1 - len(y) % 2)))\\n    if len(y) < w:\\n        return y\\n    return savgol_filter(y, w, poly, mode=\\\"interp\\\")\\n\"}")
for rel, content in payload.items():
    p = ROOT / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding='utf-8')
print('wrote', len(payload), 'files')


In [ ]:
from pathlib import Path
import json
import sys
BUNDLE = Path.cwd().resolve()
sys.path.insert(0, str(BUNDLE))
from run_kaggle_pipeline import resolve_data_dir, run_pipeline
data_dir = resolve_data_dir(None)
if Path('/kaggle/input/rogii-synthetic-trace-smoke-data').is_dir():
    data_dir = Path('/kaggle/input/rogii-synthetic-trace-smoke-data')
work_dir = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('output')
metrics = run_pipeline(
    data_dir,
    work_dir,
    rmse_threshold=12.0,
    submit=True,
    episodic_episodes=2,
    episodic_epochs=30,
)
print(json.dumps(metrics, indent=2))
